
---

### Declaration of Usage of Generative AI
In this work, Generative AI was used to post-edit the wording of **some** written paragraphs and comments, including the ones in the final report, to improve their clarity and style, as English is not the native language of any of the group members. The underlying ideas and content remain entirely our own. In terms of coding, AI assistance was used **solely** for debugging purposes when exceptions occurred in the code. **No** generative AI tools were used as a learning assistant during the completion of this exercise.

---



In [ ]:
import re
import warnings

_old = warnings.showwarning

PATTERNS = [
    r"datetime\.datetime\.utcnow\(\) is deprecated",
    r"This process .* is multi-threaded, use of fork\(\) may lead to deadlocks",
]

def _showwarning(message, category, filename, lineno, file=None, line=None):
    text = str(message)
    if category is DeprecationWarning and any(re.search(p, text) for p in PATTERNS):
        return
    return _old(message, category, filename, lineno, file=file, line=line)

warnings.showwarning = _showwarning


In [ ]:
!pip install gensim
!pip install contextualized_topic_models
!pip install pyldavis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 96.7 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 46.1 MB/s eta 0:00:00


## General Instructions

1. Perform Topic Modeling using LDA and CTM on the three time frames: before 1990, 1990-2009 and 2010 onwards.
2. Experiment with a) different preprocessing functions and b) varying number of topics.
3. Annotate the topics.
4. Answer the questions marked with 📝❓ in your lab report at the end of this notebook  

## Import Libraries

In [ ]:
import os
import torch
import random
import numpy as np

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    os.environ["PYTHONHASHSEED"] = str(seed)

    print(f"[Seeded everything with seed={seed}]")

seed = 42
seed_everything(seed=seed)

[Seeded everything with seed=42]


In [ ]:
import io
import re
import csv
import gzip
import urllib
import random
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from IPython.display import display
from typing import List, Sequence, Any, Tuple

pd.set_option('display.max_rows', 500)

## Download Dataset

In [ ]:
url_before_1990 = 'https://drive.google.com/file/d/1o_IeJCqvDLH5xgjYYuEHoPuPjF7SYvwR/view?usp=drive_link'
url_from_1990_to_2009 = 'https://drive.google.com/file/d/1Q31iYPxlcsvB0nwGter3RDfbhVRtV2yI/view?usp=drive_link'
url_from_2010 = 'https://drive.google.com/file/d/1s7pLqaiMVxM0M4WBKgZpBxNDFKXeQ47x/view?usp=drive_link'

In [ ]:
# Function to download data given a google drive url - Returns a list
import requests

def download_text_file_from_drive(drive_url):
    try:
        file_id = drive_url.split('/d/')[1].split('/')[0]
    except IndexError:
        raise ValueError("Invalid Google Drive URL format. Ensure it includes '/d/<file_id>/'.")

    download_url = f"https://drive.google.com/uc?id={file_id}&export=download"

    response = requests.get(download_url)
    if response.status_code != 200:
        raise RuntimeError(f"Failed to download file. HTTP Status Code: {response.status_code}")

    content = response.text
    titles_year = content.splitlines()
    titles = [x.split(',')[0] for x in titles_year]
    return titles

In [ ]:
titles_before_1990 = download_text_file_from_drive(url_before_1990)
titles_from_1990_to_2009 = download_text_file_from_drive(url_from_1990_to_2009)
titles_from_2010 = download_text_file_from_drive(url_from_2010)

# Check the length of downloaded data
print(len(titles_before_1990))
print(len(titles_from_1990_to_2009))
print(len(titles_from_2010))

# Check the first element of each list
# Elements in the list are of the format - paper_title, year
print(titles_before_1990[0])
print(titles_from_1990_to_2009[0])
print(titles_from_2010[0])

40000
243581
582378
An Introduction to Mathematical Taxonomy
The Future of Classic Data Administration: Objects + Databases + CASE
E. W. Dijkstra Archive: The manuscripts of Edsger W. Dijkstra 1930-2002


## Preprocessing Functions

*Optionally, you can write the preprocessing functions for LDA here or use inbuilt sklearn functionalities for preprocessing while performing LDA*

*For CTMs, it is recommended that you preprocess the dataset only for creating Bag of Words, while the embeddings are generated without doing any preprocessing. This will ensure that better quality embeddings are generated as more context is present, without the vocabulary size becoming huge. You can refer to authors' proposed preprocessing implementation [here](https://github.com/MilaNLProc/contextualized-topic-models?tab=readme-ov-file#preprocessing)*

https://scikit-learn.org/stable/auto_examples/applications/plot_topics_extraction_with_nmf_lda.html#sphx-glr-auto-examples-applications-plot-topics-extraction-with-nmf-lda-py

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download the NLTK stopwords corpus
nltk.download('stopwords')

# Download the NLTK WordNet corpus (used for lemmatization)
nltk.download('wordnet')

# Create a list of English stopwords
stop_words = list(stopwords.words('english'))

# Initialize a WordNet-based lemmatizer
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
def preprocess1(titles: List[str]) -> List[List[str]]:
  """
  Basic Preprocessing:
    - Lowercase;
    - Remove non-alphabetic characters;
    - Remove stopwords or words shorter than 3 characters.
  """

  clean_titles = []

  for title in titles:

    # Remove special characters and numbers, convert to lower case
    title = re.sub("[^a-zA-Z]", " ", title.lower())
    tokens = title.split()

    # Remove stopwords and short words
    tokens = [t for t in tokens if t not in stop_words and len(t) >= 3]

    clean_titles.append(tokens)

  return clean_titles

In [ ]:
def preprocess2(titles: List[str]) -> List[List[str]]:
  """
  Advanced Preprocessing:
    - Lowercase;
    - Remove non-alphabetic characters;
    - Remove stopwords;
    - Perform lemmatization.
  """

  clean_titles = []

  for title in titles:

    # Remove special characters and numbers, convert to lower case
    title = re.sub("[^a-zA-Z]", " ", title.lower())
    tokens = title.split()

    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stopwords.words('english')]

    clean_titles.append(tokens)

  return clean_titles

## LDA

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

num_lda_topics = 5 # min number of topics

In [ ]:
import gensim
import gensim.corpora as corpora
from gensim.models import LdaMulticore, CoherenceModel


def lda_pipeline(clean_titles: List[List[str]],
                 num_topics: int = num_lda_topics,
                 random_state: int = 42,
                 passes: int = 10) -> pd.DataFrame:
    """
    Runs a complete LDA pipeline using Gensim.

    1. Creates Gensim Dictionary and Corpus.
    2. Trains Gensim LDA Model.
    3. Calculates Coherence Score (c_v).

    Args:
        clean_titles (list): List of lists of apreprocessednd tokenized strings.
        num_topics (int): Number of topics to find.
        random_state (int): Seed for reproducibility.
        passes (int): Number of passes through the corpus during training.
    """

    # Create a dictionary
    id2word = corpora.Dictionary(clean_titles)

    # Create a bag-of-words
    corpus = [id2word.doc2bow(text) for text in clean_titles]

    # Use LdaMulticore, which is faster for large datasets
    print(f"Training Gensim LDA with {num_topics} topics...")
    model = LdaMulticore(corpus=corpus,
                         id2word=id2word,
                         num_topics=num_topics,
                         random_state=random_state,
                         passes=passes,
                         workers=2)

    # Calculate coherence score
    print("Calculating Coherence Score...")
    coherence_model = CoherenceModel(model=model,
                                     texts=clean_titles,
                                     dictionary=id2word,
                                     coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    print(f"\nCoherence Score: {coherence_score}")

    # Get the topics, the words and their probabilities
    topics = model.show_topics(num_topics=num_topics,
                               num_words=10,
                               formatted=False)

    # Create rows of the dataframe
    rows = []
    for topic_id, word_probs in topics:
        for word, score in word_probs:
            rows.append({"topic_id": topic_id,
                         "word": word,
                         "score": score})

    # Combine them in a dataframe
    topic_word_df = (pd.DataFrame(rows).sort_values(["topic_id", "score"],
                                                   ascending=[True, False])
                                       .set_index(['topic_id', 'word']))

    return topic_word_df

### Before the 1990s:

In [ ]:
# Preprocess 1
clean_titles1_before_1990 = preprocess1(titles=titles_before_1990)

In [ ]:
# Perform LDA with num_lda_topics = 5 for Preprocess 1
lda_pipeline(clean_titles=clean_titles1_before_1990,
             num_topics=5)

Training Gensim LDA with 5 topics...
Calculating Coherence Score...

Coherence Score: 0.38863099981578025


score
topic_id word                 
0        algorithm    0.016925
         problem      0.014702
         problems     0.011342
         note         0.009791
         linear       0.009573
         algorithms   0.008306
         graphs       0.008008
         method       0.007957
         two          0.007587
         functions    0.006289
1        systems      0.032687
         system       0.026790
         control      0.019733
         design       0.014964
         analysis     0.013587
         data         0.013546
         software     0.010595
         time         0.009769
         computer     0.009629
         model        0.009532
2        programming  0.019568
         theory       0.014547
         logic        0.012379
         recognition  0.011887
         language     0.008987
         pattern      0.008276
         computer     0.007778
         comments     0.005884
         analysis     0.005502
         using        0.005495
3        computer     0.021874
         review       0.016540
         science      0.011392
         security     0.009196
         research     0.008055
         book         0.007396
         computers    0.007130
         chess        0.007090
         data         0.006198
         information  0.005858
4        uuml         0.026655
         der          0.025129
         auml         0.016512
         und          0.015673
         von          0.015598
         zur          0.015198
         die          0.012068
         sub          0.009897
         ein          0.009216
         ouml         0.009207

In [ ]:
# Perform LDA with num_lda_topics > 5 for Preprocess 1
lda_pipeline(clean_titles=clean_titles1_before_1990,
             num_topics=10)

Training Gensim LDA with 10 topics...
Calculating Coherence Score...

Coherence Score: 0.39618668682481883


score
topic_id word                    
0        languages       0.021358
         programming     0.021006
         programs        0.020250
         methods         0.017130
         equations       0.016270
         language        0.015134
         program         0.013096
         free            0.010722
         note            0.009833
         generation      0.009533
1        system          0.058714
         systems         0.033756
         control         0.018176
         review          0.016143
         design          0.016080
         software        0.015909
         analysis        0.014327
         information     0.014243
         model           0.012563
         data            0.011602
2        theorem         0.015131
         retrieval       0.014547
         set             0.012604
         information     0.011499
         matrix          0.009783
         theory          0.009530
         boolean         0.009068
         points          0.008958
         games           0.007733
         artificial      0.007391
3        time            0.016240
         sets            0.014424
         finite          0.013621
         order           0.009736
         method          0.009648
         model           0.009617
         first           0.008733
         functions       0.008214
         estimation      0.007848
         models          0.007678
4        uuml            0.050441
         der             0.044522
         und             0.030807
         auml            0.029519
         von             0.029449
         zur             0.026070
         die             0.022041
         ouml            0.019134
         ein             0.016612
         mit             0.015113
5        systems         0.031620
         linear          0.028366
         control         0.028330
         optimal         0.020743
         problems        0.020143
         algorithms      0.020125
         algorithm       0.019075
         problem         0.017254
         time            0.013914
         programming     0.012228
6        logic           0.031381
         complexity      0.022955
         theory          0.019969
         space           0.014465
         automata        0.013993
         matrices        0.009887
         machines        0.009239
         order           0.009150
         computational   0.008361
         finite          0.008142
7        using           0.020551
         algorithm       0.017454
         recognition     0.016696
         new             0.015025
         image           0.014344
         sub             0.012490
         pattern         0.011465
         high            0.010454
         method          0.009548
         digital         0.008787
8        comments        0.017064
         scientific      0.015690
         local           0.015505
         noise           0.012441
         analysis        0.010563
         area            0.010254
         special         0.008740
         code            0.008260
         log             0.008238
         considerations  0.007544
9        computer        0.056957
         data            0.024549
         design          0.022037
         systems         0.015103
         science         0.012540
         information     0.012348
         software        0.010207
         security        0.008809
         research        0.008641
         memory          0.008089

----

In [ ]:
# Preprocess 2
clean_titles2_before_1990 = preprocess2(titles=titles_before_1990)

In [ ]:
# Perform LDA with num_lda_topics = 5 for Preprocess 2
lda_pipeline(clean_titles=clean_titles2_before_1990,
             num_topics=5)

Training Gensim LDA with 5 topics...
Calculating Coherence Score...

Coherence Score: 0.3818045648864606


score
topic_id word                 
0        computer     0.031596
         system       0.026997
         data         0.017788
         information  0.013213
         design       0.013014
         software     0.011536
         program      0.010669
         language     0.010502
         programming  0.010436
         processing   0.009776
1        system       0.051715
         control      0.026218
         model        0.014653
         time         0.014029
         analysis     0.012050
         using        0.009249
         application  0.009109
         based        0.008871
         approach     0.008758
         linear       0.008603
2        set          0.024361
         der          0.018199
         uuml         0.014357
         von          0.012014
         und          0.011503
         zur          0.010618
         die          0.008005
         auml         0.006492
         impact       0.005466
         degree       0.005167
3        algorithm    0.027310
         problem      0.018108
         graph        0.012270
         method       0.009713
         note         0.009252
         tree         0.007915
         function     0.007140
         linear       0.006644
         two          0.006619
         equation     0.006567
4        review       0.023325
         uuml         0.015793
         r            0.015459
         de           0.012999
         auml         0.012046
         der          0.011107
         ouml         0.011086
         l            0.011013
         f            0.010820
         pattern      0.010006

In [ ]:
# Perform LDA with num_lda_topics > 5 for Preprocess 2
lda_pipeline(clean_titles=clean_titles2_before_1990,
             num_topics=10)

Training Gensim LDA with 10 topics...
Calculating Coherence Score...

Coherence Score: 0.3642622173627289


score
topic_id word                    
0        computer        0.094434
         information     0.034091
         science         0.022688
         retrieval       0.016773
         graphic         0.015290
         system          0.013240
         research        0.012626
         technology      0.010856
         network         0.009587
         chess           0.009511
1        system          0.074743
         data            0.038359
         design          0.027282
         control         0.025549
         software        0.015937
         based           0.014503
         model           0.013079
         development     0.012046
         dynamic         0.011319
         application     0.011230
2        theory          0.054270
         programming     0.035430
         language        0.028597
         logic           0.016799
         automaton       0.014908
         model           0.014412
         type            0.012767
         set             0.011498
         result          0.009321
         functional      0.008860
3        algorithm       0.030927
         problem         0.020863
         tree            0.020109
         program         0.013338
         note            0.013052
         bound           0.012288
         number          0.011329
         search          0.010357
         fast            0.010043
         theorem         0.009418
4        time            0.056800
         system          0.039180
         network         0.028323
         high            0.014652
         real            0.013892
         microprocessor  0.011732
         signal          0.009716
         analysis        0.009662
         level           0.009488
         discrete        0.009287
5        performance     0.028102
         system          0.018721
         parallel        0.017807
         algorithm       0.017049
         digital         0.016974
         filter          0.014821
         simulation      0.014247
         analysis        0.013409
         line            0.013248
         order           0.012225
6        uuml            0.043555
         der             0.038623
         r               0.028108
         und             0.026299
         auml            0.025908
         von             0.025800
         zur             0.023079
         f               0.020608
         die             0.019152
         ouml            0.016670
7        method          0.025659
         linear          0.018210
         system          0.016770
         model           0.013756
         using           0.013426
         recognition     0.013393
         pattern         0.011969
         algorithm       0.011813
         equation        0.011413
         problem         0.011043
8        review          0.023074
         problem         0.021612
         set             0.020558
         algorithm       0.018827
         sub             0.018215
         sup             0.016195
         n               0.015889
         function        0.015596
         complexity      0.013247
         group           0.011317
9        graph           0.047078
         de              0.017694
         eacute          0.014043
         scientific      0.011587
         maximum         0.011241
         complete        0.010738
         problem         0.010369
         query           0.008229
         cycle           0.008151
         l               0.008072

### From 1990 to 2009:

*Add your code for topic modelling the period from 1990 to 2009 here - similar to what you did for before 1990s*

In [ ]:
# Preprocess 1
clean_titles1_from_1990_to_2009 = preprocess1(titles=titles_from_1990_to_2009)

In [ ]:
# Perform LDA with num_lda_topics = 5 for Preprocess 1
lda_pipeline(clean_titles=clean_titles1_from_1990_to_2009,
             num_topics=5,
             passes=5)

Training Gensim LDA with 5 topics...
Calculating Coherence Score...

Coherence Score: 0.35510118417627956


score
topic_id word                 
0        system       0.010622
         simulation   0.009142
         study        0.008342
         control      0.007954
         using        0.007516
         human        0.007196
         robot        0.007037
         virtual      0.006738
         model        0.006470
         memory       0.006344
1        information  0.023452
         systems      0.011666
         software     0.011049
         management   0.010571
         case         0.009591
         research     0.009481
         based        0.009470
         web          0.009122
         science      0.008287
         system       0.008270
2        time         0.011920
         linear       0.011907
         two          0.010263
         using        0.010128
         method       0.008013
         analysis     0.007990
         order        0.007719
         discrete     0.007227
         algorithm    0.007157
         functions    0.006231
3        networks     0.021826
         based        0.020272
         systems      0.019201
         control      0.016809
         using        0.016005
         analysis     0.013088
         network      0.012864
         time         0.011968
         neural       0.011624
         performance  0.010234
4        problem      0.023673
         sub          0.020286
         problems     0.011598
         graphs       0.011515
         finite       0.010626
         algorithm    0.009263
         random       0.008442
         graph        0.008385
         method       0.008347
         type         0.007322

In [ ]:
# Perform LDA with num_lda_topics > 5 for Preprocess 1
lda_pipeline(clean_titles=clean_titles1_from_1990_to_2009,
             num_topics=10,
             passes=5)

Training Gensim LDA with 10 topics...
Calculating Coherence Score...

Coherence Score: 0.3749715050808627


score
topic_id word                   
0        using          0.018368
         model          0.014712
         memory         0.013493
         dimensional    0.013468
         simulation     0.011008
         analysis       0.010935
         recognition    0.010667
         human          0.010648
         large          0.010539
         based          0.010387
1        performance    0.044230
         model          0.036064
         analysis       0.018848
         high           0.015213
         level          0.013927
         using          0.009437
         data           0.009045
         one            0.008973
         prediction     0.008922
         filter         0.008027
2        graphs         0.022980
         self           0.022904
         properties     0.013358
         class          0.011402
         frequency      0.009439
         convex         0.008674
         wavelet        0.008575
         controller     0.008314
         codes          0.007749
         functions      0.007707
3        sub            0.033251
         sup            0.025320
         object         0.020511
         model          0.014644
         fault          0.013817
         analysis       0.013199
         oriented       0.011797
         using          0.011436
         brain          0.010719
         selection      0.010336
4        algorithm      0.031869
         data           0.029873
         algorithms     0.025894
         problem        0.025550
         parallel       0.016493
         optimization   0.015570
         problems       0.014454
         search         0.010571
         efficient      0.009767
         graphs         0.009722
5        based          0.027024
         using          0.024194
         image          0.019937
         fast           0.018158
         estimation     0.015750
         efficient      0.013058
         algorithm      0.012787
         signal         0.011286
         path           0.010634
         digital        0.010363
6        theory         0.032638
         complexity     0.024613
         output         0.017230
         number         0.015542
         editorial      0.014043
         issue          0.013279
         international  0.012716
         lower          0.011408
         evolution      0.011011
         bounds         0.010996
7        information    0.027480
         software       0.016427
         based          0.016360
         systems        0.016299
         design         0.016032
         system         0.015185
         computer       0.012156
         research       0.012151
         development    0.012095
         web            0.010765
8        networks       0.049424
         control        0.033476
         network        0.029363
         neural         0.027067
         systems        0.020149
         time           0.016496
         dynamic        0.015780
         based          0.015158
         distributed    0.014978
         mobile         0.013065
9        systems        0.044686
         time           0.034792
         linear         0.020843
         control        0.020463
         nonlinear      0.019152
         finite         0.016957
         order          0.015498
         method         0.013431
         discrete       0.012294
         equations      0.011741

---

In [ ]:
# Preprocess 2
clean_titles2_from_1990_to_2009 = preprocess2(titles=titles_from_1990_to_2009)

In [ ]:
# Perform LDA with num_lda_topics = 5 for Preprocess 2
lda_pipeline(clean_titles=clean_titles2_from_1990_to_2009,
             num_topics=5,
             passes=5)

Training Gensim LDA with 5 topics...
Calculating Coherence Score...

Coherence Score: 0.3413354179321305


score
topic_id word                   
0        system         0.054127
         control        0.025678
         design         0.021175
         based          0.015098
         time           0.014449
         model          0.012507
         software       0.010446
         approach       0.009793
         analysis       0.009690
         sub            0.009639
1        k              0.012777
         note           0.012489
         domain         0.011951
         e              0.011892
         square         0.008102
         r              0.007695
         l              0.007052
         amp            0.006832
         n              0.006782
         least          0.006396
2        algorithm      0.024168
         problem        0.020670
         method         0.014931
         graph          0.013255
         parallel       0.010664
         linear         0.009264
         equation       0.007659
         sup            0.007381
         finite         0.007307
         two            0.007163
3        network        0.021061
         information    0.020194
         research       0.011397
         based          0.011075
         service        0.009696
         system         0.008801
         science        0.008489
         technology     0.008014
         communication  0.007879
         library        0.007738
4        data           0.018759
         using          0.017428
         analysis       0.016567
         based          0.016153
         study          0.014779
         network        0.013045
         model          0.012581
         estimation     0.009742
         neural         0.008138
         web            0.008035

In [ ]:
# Perform LDA with num_lda_topics > 5 for Preprocess 2
lda_pipeline(clean_titles=clean_titles2_from_1990_to_2009,
             num_topics=10,
             passes=5)

Training Gensim LDA with 10 topics...
Calculating Coherence Score...

Coherence Score: 0.3697822333082941


score
topic_id word                 
0        system       0.079141
         control      0.042040
         time         0.030618
         design       0.020602
         based        0.018949
         model        0.017440
         dynamic      0.014706
         approach     0.013014
         fuzzy        0.012691
         real         0.012513
1        graph        0.046690
         sub          0.029797
         citation     0.021043
         journal      0.020436
         number       0.018217
         bound        0.018152
         delay        0.016398
         publication  0.012861
         k            0.010490
         h            0.010398
2        algorithm    0.040329
         parallel     0.031667
         tree         0.021955
         performance  0.019544
         efficient    0.018628
         learning     0.015407
         programming  0.014670
         computing    0.014116
         computer     0.013248
         program      0.012905
3        environment  0.024685
         robot        0.024338
         object       0.022011
         system       0.019013
         oriented     0.018839
         virtual      0.016702
         issue        0.016078
         human        0.014032
         output       0.013927
         mobile       0.013535
4        data         0.036197
         web          0.022802
         service      0.017304
         database     0.017231
         system       0.017011
         model        0.014427
         based        0.014108
         analysis     0.012554
         traffic      0.012140
         input        0.011158
5        network      0.084055
         neural       0.028175
         based        0.014011
         using        0.013394
         wireless     0.013233
         performance  0.012132
         protocol     0.011910
         channel      0.010238
         routing      0.010141
         scheme       0.010123
6        study        0.028161
         research     0.023817
         information  0.022729
         science      0.018984
         system       0.017923
         case         0.015840
         software     0.014956
         technology   0.012718
         development  0.012707
         process      0.012216
7        two          0.020750
         time         0.020698
         analysis     0.019827
         algorithm    0.019686
         function     0.016593
         method       0.016032
         estimation   0.015957
         linear       0.015553
         filter       0.012511
         model        0.011735
8        using        0.030375
         based        0.029651
         image        0.021210
         pattern      0.014523
         library      0.013732
         data         0.012696
         model        0.012629
         retrieval    0.011236
         analysis     0.010708
         method       0.010553
9        problem      0.068290
         method       0.030702
         equation     0.024553
         sup          0.023409
         finite       0.023392
         solution     0.017609
         path         0.015678
         element      0.014624
         complexity   0.009451
         n            0.009438

### From 2010 onwards:

*Add your code for topic modelling the period from 2010 onwards here - similar to what you did for before 1990s*

In [ ]:
# Preprocess 1
clean_titles1_from_2010 = preprocess1(titles=titles_from_2010)

In [ ]:
# Perform LDA with num_lda_topics = 5 for Preprocess 1
lda_pipeline(clean_titles=clean_titles1_from_2010,
             num_topics=5,
             passes=1)

Training Gensim LDA with 5 topics...
Calculating Coherence Score...

Coherence Score: 0.36982748822029643


score
topic_id word                    
0        based           0.017281
         networks        0.013984
         analysis        0.013664
         network         0.009372
         using           0.009034
         method          0.008709
         neural          0.007280
         algorithm       0.007096
         finite          0.006635
         graphs          0.006390
1        learning        0.045675
         based           0.028469
         using           0.018326
         deep            0.016394
         detection       0.013497
         network         0.011473
         machine         0.011418
         classification  0.009908
         recognition     0.008345
         human           0.008258
2        based           0.024272
         time            0.017212
         networks        0.015816
         algorithm       0.012486
         wireless        0.010910
         multi           0.010698
         using           0.010403
         method          0.009887
         distributed     0.009862
         systems         0.009579
3        research        0.016398
         information     0.012055
         study           0.010855
         data            0.010114
         management      0.008060
         case            0.007740
         analysis        0.007490
         cloud           0.006794
         science         0.006792
         based           0.006377
4        systems         0.027103
         based           0.022531
         control         0.022246
         system          0.017106
         design          0.012526
         model           0.011255
         using           0.010252
         multi           0.009864
         analysis        0.008962
         fuzzy           0.008020

In [ ]:
# Perform LDA with num_lda_topics > 5 for Preprocess 1
lda_pipeline(clean_titles=clean_titles1_from_2010,
             num_topics=10,
             passes=1)

Training Gensim LDA with 10 topics...
Calculating Coherence Score...

Coherence Score: 0.31111371626827194


score
topic_id word                    
0        analysis        0.023886
         finite          0.018323
         graphs          0.015008
         delay           0.011156
         study           0.008947
         academic        0.008800
         complexity      0.008380
         element         0.008283
         media           0.008189
         functional      0.007471
1        learning        0.076718
         based           0.035788
         deep            0.026775
         detection       0.024043
         using           0.022942
         machine         0.019593
         recognition     0.015307
         classification  0.012815
         citation        0.010412
         model           0.009230
2        control         0.031033
         based           0.022374
         systems         0.019177
         large           0.014443
         scale           0.012712
         adaptive        0.012318
         multi           0.012025
         robot           0.011635
         method          0.010864
         mimo            0.010860
3        based           0.034447
         network         0.019734
         using           0.015996
         image           0.014663
         via             0.010555
         multi           0.010168
         retrieval       0.010088
         graph           0.010064
         attention       0.009560
         efficient       0.009546
4        neural          0.035687
         network         0.031016
         sub             0.024098
         using           0.021259
         based           0.017688
         sensing         0.013805
         integrated      0.012947
         bibliometric    0.012903
         convolutional   0.012149
         networks        0.009263
5        systems         0.016351
         method          0.014320
         model           0.012944
         equations       0.012825
         problems        0.010930
         nonlinear       0.010927
         stochastic      0.010880
         two             0.010741
         open            0.010203
         control         0.009400
6        based           0.032176
         algorithm       0.028700
         networks        0.021280
         optimization    0.019086
         using           0.011551
         selection       0.011420
         method          0.010355
         scheme          0.010177
         multi           0.009794
         problem         0.009231
7        time            0.045395
         based           0.026862
         systems         0.024173
         data            0.020285
         computing       0.019601
         decision        0.012587
         system          0.012413
         review          0.012036
         real            0.011284
         improved        0.011036
8        impact          0.026488
         web             0.018647
         power           0.014195
         semantic        0.012658
         system          0.011663
         journal         0.011546
         intelligent     0.011473
         systems         0.010266
         transmission    0.009603
         cooperative     0.009392
9        networks        0.019782
         research        0.018179
         based           0.013242
         information     0.012623
         study           0.011872
         data            0.011126
         management      0.011075
         wireless        0.010756
         analysis        0.010539
         energy          0.010326

---

In [ ]:
# Preprocess 2
clean_titles2_from_2010 = preprocess2(titles=titles_from_2010)

In [ ]:
# Perform LDA with num_lda_topics = 5 for Preprocess 2
lda_pipeline(clean_titles=clean_titles2_from_2010,
             num_topics=5,
             passes=1)

Training Gensim LDA with 5 topics...
Calculating Coherence Score...

Coherence Score: 0.33831935214528747


score
topic_id word                   
0        data           0.024182
         analysis       0.016265
         based          0.015591
         system         0.013578
         research       0.012892
         study          0.012204
         using          0.010504
         cloud          0.009722
         model          0.007681
         approach       0.007649
1        network        0.068437
         based          0.022145
         neural         0.016460
         wireless       0.011696
         communication  0.010026
         mobile         0.009367
         sensor         0.008970
         social         0.008682
         user           0.007039
         using          0.006904
2        system         0.048567
         control        0.023404
         time           0.022404
         design         0.017425
         based          0.017333
         model          0.010723
         fuzzy          0.009327
         multi          0.008558
         optimal        0.008106
         dynamic        0.007741
3        based          0.025391
         algorithm      0.019119
         method         0.018825
         using          0.015681
         image          0.010640
         equation       0.009411
         problem        0.009097
         scheme         0.008070
         estimation     0.007250
         sub            0.007003
4        learning       0.031564
         graph          0.015196
         based          0.013517
         using          0.012449
         model          0.012386
         deep           0.011939
         detection      0.007840
         machine        0.007364
         journal        0.007104
         library        0.006311

In [ ]:
# Perform LDA with num_lda_topics > 5 for Preprocess 2
lda_pipeline(clean_titles=clean_titles2_from_2010,
             num_topics=10,
             passes=1)

Training Gensim LDA with 10 topics...
Calculating Coherence Score...

Coherence Score: 0.3229446900127079


score
topic_id word                    
0        analysis        0.023749
         citation        0.021959
         method          0.018300
         using           0.018200
         based           0.016389
         data            0.011484
         identification  0.010497
         model           0.009200
         estimation      0.008975
         automatic       0.008971
1        network         0.062098
         based           0.022786
         wireless        0.018103
         analysis        0.014498
         social          0.014218
         sensor          0.013962
         mobile          0.013442
         communication   0.013311
         information     0.009919
         user            0.009109
2        system          0.066903
         control         0.041561
         time            0.036392
         based           0.019639
         design          0.014821
         model           0.014688
         fuzzy           0.013303
         decision        0.013194
         adaptive        0.012010
         optimal         0.011968
3        method          0.034337
         algorithm       0.030218
         problem         0.025130
         based           0.022577
         equation        0.016556
         novel           0.013618
         optimization    0.012815
         scheme          0.011429
         finite          0.010563
         adaptive        0.009995
4        model           0.019394
         using           0.015716
         issue           0.012380
         simulation      0.011223
         robot           0.011008
         based           0.010764
         editorial       0.010545
         human           0.009984
         special         0.009925
         flow            0.008292
5        high            0.016493
         channel         0.015777
         field           0.014511
         low             0.012070
         rate            0.011386
         brain           0.011197
         using           0.010008
         based           0.009433
         manufacturing   0.009171
         p               0.008944
6        review          0.019477
         system          0.019351
         web             0.018239
         software        0.016625
         science         0.015870
         open            0.014985
         sup             0.014945
         virtual         0.014330
         computer        0.014005
         intelligence    0.013433
7        learning        0.047522
         based           0.045823
         network         0.044224
         neural          0.024715
         using           0.022930
         deep            0.020244
         detection       0.020051
         algorithm       0.016379
         efficient       0.012732
         multi           0.012692
8        data            0.044440
         research        0.024945
         system          0.018592
         based           0.016583
         cloud           0.014759
         information     0.011882
         management      0.010778
         scientific      0.010140
         computing       0.009928
         technology      0.009859
9        study           0.031412
         journal         0.020961
         case            0.019308
         graph           0.018507
         impact          0.016130
         selection       0.013701
         publication     0.012847
         number          0.012766
         university      0.011155
         product         0.010292

📝❓ For each period, assign a name to each generated topic based on the topic’s top words. List all topic names in your report. If a topic is incoherent to the degree that no common theme is detectable, you can just mark it as incoherent (i.e., no need to name a topic that does not exist).


✅ Assigning name to each generated topic:
## **Before 1990**

| Preprocessing   | Number of Topics | Topic Index | Topic Name                           |
|------------------|------------------|-------------|---------------------------------------|
| Preprocessing 1 | 5                | 0           | Algorithms and Problem Solving    |
| Preprocessing 1 | 5                | 1           | Systems Engineering and Control     |
| Preprocessing 1 | 5                | 2           | Programming Languages and Logic |
| Preprocessing 1 | 5                | 3           | Computer Science Research and Applications |
| Preprocessing 1 | 5                | 4           | **Incoherent**           |
| Preprocessing 1 | 10               | 0           | Programs, Methods and Numerical Equations                   |
| Preprocessing 1 | 10               | 1           | Systems Design, Information and Software                 |
| Preprocessing 1 | 10               | 2           | Discrete Math               |
| Preprocessing 1 | 10               | 3           | Finite Set Theory       |
| Preprocessing 1 | 10               | 4           | **Incoherent**                  |
| Preprocessing 1 | 10               | 5           | Linear Systems and Optimal Control                         |
| Preprocessing 1 | 10               | 6           | Logic, Complexity Theory and Automata             |
| Preprocessing 1 | 10               | 7           | Pattern Recognition and Detection                  |
| Preprocessing 1 | 10               | 8           | **Incoherent**               |
| Preprocessing 1 | 10               | 9           | Computer Science Research and Information Systems                    |
| Preprocessing 2 | 5                | 0           | Computer Systems and Software Engineering                         |
| Preprocessing 2 | 5                | 1           | Control Systems Modeling and Analysis                 |
| Preprocessing 2 | 5                | 2           | **Incoherent**                  |
| Preprocessing 2 | 5                | 3           | Algorithms and Graph-Based Methods             |
| Preprocessing 2 | 5                | 4           | **Incoherent**                |
| Preprocessing 2 | 10               | 0           | Computer and Information Science                      |
| Preprocessing 2 | 10               | 1           | System Design and Software Development                     |
| Preprocessing 2 | 10               | 2           | Programming Language Theory and Formal Logic               |
| Preprocessing 2 | 10               | 3           | Algorithmic Theory                  |
| Preprocessing 2 | 10               | 4           | Networked Systems                      |
| Preprocessing 2 | 10               | 5           | Performance Analysis                  |
| Preprocessing 2 | 10               | 6           | **Incoherent**                  |
| Preprocessing 2 | 10               | 7           | Linear Methods for Pattern Recognition                |
| Preprocessing 2 | 10               | 8           | Theoretical Computer Science                |
| Preprocessing 2 | 10               | 9           | **Incoherent**              |

---


## **1990-2009**

| Preprocessing   | Number of Topics | Topic Index | Topic Name                           |
|------------------|------------------|-------------|---------------------------------------|
| Preprocessing 1 | 5                | 0           | Human-Robot Control and Simulation                |
| Preprocessing 1 | 5                | 1           | Information Systems and Software Management                     |
| Preprocessing 1 | 5                | 2           | Linear Methods                |
| Preprocessing 1 | 5                | 3           | Networked Control Systems        |
| Preprocessing 1 | 5                | 4           | Graph Algorithms               |
| Preprocessing 1 | 10               | 0           | **Incoherent**           |
| Preprocessing 1 | 10               | 1           | Performance Modeling           |
| Preprocessing 1 | 10               | 2           | Graph Properties              |
| Preprocessing 1 | 10               | 3           | **Incoherent**                      |
| Preprocessing 1 | 10               | 4           | Algorithms, Optimization and Graph Problems                       |
| Preprocessing 1 | 10               | 5           | Image and Signal Processing Algorithms            |
| Preprocessing 1 | 10               | 6           | Complexity Theory and Bounds                     |
| Preprocessing 1 | 10               | 7           | Information Systems and Software Design                     |
| Preprocessing 1 | 10               | 8           | Distributed Control Systems                |
| Preprocessing 1 | 10               | 9           | Control Theory and Time-Driven Systems                   |
| Preprocessing 2 | 5                | 0           | System Control and Software Design                   |
| Preprocessing 2 | 5                | 1           | **Incoherent**                    |
| Preprocessing 2 | 5                | 2           | Graph and Parallel Design                   |
| Preprocessing 2 | 5                | 3           | Networked Information Services          |
| Preprocessing 2 | 5                | 4           | Neural Networks and Data Analysis                      |
| Preprocessing 2 | 10               | 0           | Control Systems              |
| Preprocessing 2 | 10               | 1           | Graph Theory and Publication Metadata                   |
| Preprocessing 2 | 10               | 2           |  Parallel Algorithms                   |
| Preprocessing 2 | 10               | 3           | Robotics in Virtual Environments                |
| Preprocessing 2 | 10               | 4           | Web Services                      |
| Preprocessing 2 | 10               | 5           | Neural Networks              |
| Preprocessing 2 | 10               | 6           | Information Systems Research                    |
| Preprocessing 2 | 10               | 7           | Time Series Analysis                    |
| Preprocessing 2 | 10               | 8           | Image-Based Retrieval                 |
| Preprocessing 2 | 10               | 9           | Finite Methods for Equation Solving              |

---

## **2010 Onwards**

| Preprocessing   | Number of Topics | Topic Index | Topic Name                           |
|------------------|------------------|-------------|---------------------------------------|
| Preprocessing 1 | 5                | 0           | Network and Graph Analysis   |
| Preprocessing 1 | 5                | 1           | Deep Learning            |
| Preprocessing 1 | 5                | 2           | Wireless Networks              |
| Preprocessing 1 | 5                | 3           | Information Systems Research and Data Management        |
| Preprocessing 1 | 5                | 4           | Control System Design        |
| Preprocessing 1 | 10               | 0           | Graph Analysis                      |
| Preprocessing 1 | 10               | 1           | Deep Learning                 |
| Preprocessing 1 | 10               | 2           | Adaptive Control Systems                 |
| Preprocessing 1 | 10               | 3           | Image Retrieval with Neural Networks              |
| Preprocessing 1 | 10               | 4           | Neural Networks                       |
| Preprocessing 1 | 10               | 5           | **Incoherent**                 |
| Preprocessing 1 | 10               | 6           | Network Optimization              |
| Preprocessing 1 | 10               | 7           | Real-Time Decision Systems                   |
| Preprocessing 1 | 10               | 8           | **Incoherent**               |
| Preprocessing 1 | 10               | 9           | Wireless Networks Research                      |
| Preprocessing 2 | 5                | 0           | Data Analysis           |
| Preprocessing 2 | 5                | 1           | Neural Networks                    |
| Preprocessing 2 | 5                | 2           | Control Systems             |
| Preprocessing 2 | 5                | 3           | **Incoherent**                      |
| Preprocessing 2 | 5                | 4           | Deep Learning on Graphs              |
| Preprocessing 2 | 10               | 0           | Citation Analysis  |
| Preprocessing 2 | 10               | 1           | Wireless Networks                      |
| Preprocessing 2 | 10               | 2           | Control Systems                  |
| Preprocessing 2 | 10               | 3           | Optimization Methods                      |
| Preprocessing 2 | 10               | 4           | Robotics               |
| Preprocessing 2 | 10               | 5           | Signal Analysis                      |
| Preprocessing 2 | 10               | 6           | **Incoherent**                 |
| Preprocessing 2 | 10               | 7           | Neural Networks                  |
| Preprocessing 2 | 10               | 8           | Cloud Data Management                    |
| Preprocessing 2 | 10               | 9           | Bibliometrics        |

📝❓ Do the topics make sense to you? Are they coherent? Do you observe trends across different time periods? Discuss in 4-6 sentences.

✅ Across all time periods, the extracted topics are mainly coherent and consistent with major shifts in computer science.

Before 1990, the topics concentrate on foundational themes such as algorithms, graph theory, formal methods, and systems/control. Between 1990 and 2009, topics related to neural networks, robotics, and web-based services emerge more clearly, mirroring their rise. From 2010 onward, the topic space is increasingly dominated by machine learning, wireless networking, and cloud data management.


## Combined Topic Models

Method developed by [Bianchi et al. 2021](https://aclanthology.org/2021.acl-short.96/).

[A 6min presentation of the paper by one of the authors.](https://underline.io/lecture/25716-pre-training-is-a-hot-topic-contextualized-document-embeddings-improve-topic-coherence)

[Medium Blog](https://towardsdatascience.com/contextualized-topic-modeling-with-python-eacl2021-eacf6dfa576)

Code: [https://github.com/MilaNLProc/contextualized-topic-models](https://github.com/MilaNLProc/contextualized-topic-models)

Tutorial: [https://colab.research.google.com/drive/1fXJjr_rwqvpp1IdNQ4dxqN4Dp88cxO97?usp=sharing](https://colab.research.google.com/drive/1fXJjr_rwqvpp1IdNQ4dxqN4Dp88cxO97?usp=sharing)

Again, perform topic modelling for the three time periods - this time using the combined topic models (CTMs).

You can use and adapt the code from the tutorial linked above.

Use the available GPU for faster running times.

In [ ]:
from contextualized_topic_models.models.ctm import CombinedTM
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.utils.preprocessing import WhiteSpacePreprocessingStopwords

In [ ]:
num_ctm_topics = 5  # min number of topics
MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2" # Model to use for CTM

In [ ]:
# Read data
titles_before_1990 = download_text_file_from_drive(url_before_1990)
titles_from_1990_to_2009 = download_text_file_from_drive(url_from_1990_to_2009)
titles_from_2010 = download_text_file_from_drive(url_from_2010)

In [ ]:
def ctm_pipeline(preprocessed_docs: Sequence[str],
                 unpreprocessed_docs: Sequence[str],
                 n_components: int = num_ctm_topics,
                 model_name: str = MODEL_NAME,
                 num_epochs: int = 10,
                 num_data_loader_workers: int = 2) -> Tuple[CombinedTM, TopicModelDataPreparation, Any]:
    """
    Train a CombinedTM (Contextualized Topic Model) on preprocessed and raw texts.

    Args:
      preprocessed_docs: Tokenized / cleaned documents used for the bag-of-words representation.
      unpreprocessed_docs: Original (raw) documents used for contextual embeddings.
      n_components: Number of topics to learn.
      model_name: Name of the transformer model to use for contextual embeddings.
      device: Device on which to run the transformer and CTM ('cuda' or 'cpu').
      num_epochs: Number of training epochs for the CTM.
      num_data_loader_workers: Number of workers for the PyTorch DataLoader.
    """

    if len(preprocessed_docs) != len(unpreprocessed_docs):
        raise ValueError(f"preprocessed_docs (len={len(preprocessed_docs)}) and "
        f"unpreprocessed_docs (len={len(unpreprocessed_docs)}) must have the same length.")

    # Prepare data (contextual embeddings + BoW)
    print("Preparing the data...")
    tp = TopicModelDataPreparation(model_name)
    training_dataset = tp.fit(text_for_contextual=unpreprocessed_docs,
                              text_for_bow=preprocessed_docs)

    # Initialize and train CTM
    print("Training the model...")
    ctm = CombinedTM(bow_size=len(tp.vocab),
                     contextual_size=768,
                     n_components=n_components,
                     num_epochs=num_epochs)
    ctm.num_data_loader_workers = num_data_loader_workers
    ctm.fit(training_dataset)

    # Get the words in the topics
    topics = ctm.get_topic_lists(10)

    # Calculate coherence score
    print("Calculating Coherence Score...")
    tokenized_docs = [title.split() for title in preprocessed_docs]
    id2word = corpora.Dictionary(tokenized_docs)
    corpus = [id2word.doc2bow(text) for text in tokenized_docs]
    coherence_model = CoherenceModel(topics=topics,
                                     texts=tokenized_docs,
                                     dictionary=id2word,
                                     coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    print(f"\nCoherence Score: {coherence_score}")

    # Combine the topics, words in them and their probabilities
    rows = []
    tw = ctm.get_topic_word_matrix()
    for topic_id in range(tw.shape[0]):
      indices = np.argsort(-tw[topic_id])[:10]
      for idx in indices:
        rows.append({"topic_id": topic_id,
                     "word": tp.vocab[idx],
                     "score": tw[topic_id, idx]})

    # Display them as a dataframe
    topic_word_df = (pd.DataFrame(rows).sort_values(["topic_id", "score"],
                                                    ascending=[True, False])
                                       .set_index(['topic_id', 'word']))
    display(topic_word_df)

    return ctm, tp, training_dataset

In [ ]:
import pyLDAvis as vis


def vis_pipeline(ctm: CombinedTM,
                 tp: TopicModelDataPreparation,
                 training_dataset: Any) -> None:
    """
    Display an interactive pyLDAvis topic visualization for a trained CTM.

    Args:
        ctm: Trained CombinedTM model.
        tp: TopicModelDataPreparation used to build the CTM dataset.
        training_dataset: Dataset used for training/inference (for document-level stats).

    Returns:
        None. Displays the visualization in the notebook.
    """

    # Show LDAvis inside Jupyter
    vis.enable_notebook()

    # Create LDAvis inputs from CTM
    lda_vis_data = ctm.get_ldavis_data_format(tp.vocab,
                                              training_dataset,
                                              n_samples=500)

    # Build the interactive visualization object
    ctm_pd = vis.prepare(**lda_vis_data)

    # Display the interactive topic map
    display(ctm_pd)

### Before the 1990s:

In [ ]:
# Preprocess 1
preprocessed_docs1 = [' '.join(title) for title in preprocess1(titles=titles_before_1990)]
unpreprocessed_docs1 = titles_before_1990.copy()

In [ ]:
# Perform CTM with num_ctm_topics = 5 for Preprocess 1
ctm1_5, tp1_5, training_dataset1 = ctm_pipeline(preprocessed_docs=preprocessed_docs1,
                                                unpreprocessed_docs=unpreprocessed_docs1,
                                                n_components=5)

Preparing the data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/200 [00:00<?, ?it/s]

Training the model...


Epoch: [10/10]	 Seen Samples: [400000/400000]	Train Loss: 55.92994068603516	Time: 0:00:14.511883: : 10it [02:22, 14.27s/it]
100%|██████████| 625/625 [00:10<00:00, 57.74it/s]


Calculating Coherence Score...

Coherence Score: 0.4720597819414854


score
topic_id word                               
0        und                        0.469819
         von                        0.466530
         der                        0.441806
         uuml                       0.419648
         die                        0.405380
         zur                        0.395491
         des                        0.394470
         eacute                     0.385641
         ouml                       0.368769
         auml                       0.363564
1        transportverfahren         0.848217
         exklusive                  0.833601
         slot                       0.825699
         assigned                   0.816483
         beweisbare                 0.803788
         enverkehrs                 0.795077
         elektronisches             0.795056
         naturwissenschaft          0.793053
         mikroprogrammsteuerwerken  0.790613
         ils                        0.781226
2        base                       0.474652
         data                       0.471505
         system                     0.445819
         database                   0.445506
         management                 0.437235
         knowledge                  0.431427
         interactive                0.419174
         retrieval                  0.416187
         language                   0.408134
         software                   0.405725
3        estimation                 0.473155
         image                      0.448434
         adaptive                   0.445228
         solution                   0.429103
         filter                     0.426354
         equations                  0.416749
         fast                       0.415904
         nonlinear                  0.407241
         noise                      0.396377
         algorithms                 0.389712
4        spaces                     0.531221
         graphs                     0.511787
         calculus                   0.507865
         spanning                   0.502977
         set                        0.496317
         lattices                   0.489702
         designs                    0.489102
         relations                  0.487965
         equivalence                0.480534
         proof                      0.473914

In [ ]:
# Visualize 5 topics returned by CTM for Preprocess 1
vis_pipeline(ctm=ctm1_5, tp=tp1_5, training_dataset=training_dataset1)

100%|██████████| 625/625 [00:11<00:00, 55.19it/s]


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3      0.001463  0.000002       1        1  23.242482
2      0.000625 -0.000762       2        1  21.904025
0      0.000197 -0.000074       3        1  20.151684
4      0.000370  0.000894       4        1  18.601967
1     -0.002655 -0.000059       5        1  16.099841, topic_info=                            Term       Freq      Total Category  logprob  \
20321         transportverfahren  10.000000  10.000000  Default  30.0000   
6632                   exklusive  10.000000  10.000000  Default  29.0000   
20711                        und  11.000000  11.000000  Default  28.0000   
18207                       slot  10.000000  10.000000  Default  27.0000   
21445                        von  11.000000  11.000000  Default  26.0000   
12138  mikroprogrammsteuerwerken  10.000000  10.000000  Default  25.0000   
1127                    assigned  10.000000  10.000000  Default  24.0000   
5958              elektronisches  10.000000  10.000000  Default  23.0000   
9061                         ils  10.000000  10.000000  Default  22.0000   
2613                    calculus  10.000000  10.000000  Default  21.0000   
1337                aussagenkalk  10.000000  10.000000  Default  20.0000   
6273                  enverkehrs  10.000000  10.000000  Default  19.0000   
1987                  beweisbare  11.000000  11.000000  Default  18.0000   
12710       multiplexeraufwandes  10.000000  10.000000  Default  17.0000   
9445                 inhomogener  10.000000  10.000000  Default  16.0000   
18397                     spaces  11.000000  11.000000  Default  15.0000   
4802                         der  11.000000  11.000000  Default  14.0000   
8145                      graphs  11.000000  11.000000  Default  13.0000   
9434       inhaltsadressierbares  10.000000  10.000000  Default  12.0000   
12680                 multimodal  10.000000  10.000000  Default  11.0000   
12943          naturwissenschaft  11.000000  11.000000  Default  10.0000   
20989                       uuml  11.000000  11.000000  Default   9.0000   
5997                elliptischen  10.000000  10.000000  Default   8.0000   
454                      algebra  10.000000  10.000000  Default   7.0000   
17818                        set  10.000000  10.000000  Default   6.0000   
8602                  hierarchie  10.000000  10.000000  Default   5.0000   
1179              asymptotischer  11.000000  11.000000  Default   4.0000   
4850                     designs  10.000000  10.000000  Default   3.0000   
16579                  relations  10.000000  10.000000  Default   2.0000   
18407                   spanning  11.000000  11.000000  Default   1.0000   
6475                  estimation   4.463401  11.542686   Topic1  -9.4003   
7096                      filter   4.259322  11.065399   Topic1  -9.4471   
9063                       image   4.354414  11.429336   Topic1  -9.4251   
222                     adaptive   4.340479  11.431955   Topic1  -9.4283   
18328                   solution   4.271048  11.334644   Topic1  -9.4444   
17987                    signals   4.071458  10.820493   Topic1  -9.4923   
5212                    discrete   4.074216  10.868943   Topic1  -9.4916   
13387                  nonlinear   4.178687  11.194693   Topic1  -9.4663   
3179              classification   3.930853  10.571785   Topic1  -9.5274   
9000              identification   3.999462  10.770130   Topic1  -9.5101   
6320                   equations   4.218607  11.375654   Topic1  -9.4567   
13274                      noise   4.133537  11.169966   Topic1  -9.4771   
6895                        fast   4.215043  11.401005   Topic1  -9.4576   
17979                     signal   3.951267  10.728297   Topic1  -9.5222   
20249                  transform   3.933725  10.692497   Topic1  -9.5267   
18881                 stochastic   4.010946  10.915469   Topic1  -9.5072   
16298  

In [ ]:
# Perform CTM with num_ctm_topics > 5 for Preprocess 1
ctm1_10, tp1_10, training_dataset1 = ctm_pipeline(preprocessed_docs=preprocessed_docs1,
                                                  unpreprocessed_docs=unpreprocessed_docs1,
                                                  n_components=10)

Preparing the data...


Batches:   0%|          | 0/200 [00:00<?, ?it/s]

Training the model...


Epoch: [10/10]	 Seen Samples: [400000/400000]	Train Loss: 56.28736970214844	Time: 0:00:14.012077: : 10it [02:21, 14.17s/it]
100%|██████████| 625/625 [00:11<00:00, 55.31it/s]


Calculating Coherence Score...

Coherence Score: 0.46686465304587743


score
topic_id word                           
0        problems               0.750554
         solution               0.638239
         problem                0.636295
         equations              0.620006
         numerical              0.614555
         algorithms             0.555289
         algorithm              0.509062
         differential           0.492548
         solving                0.464386
         method                 0.457042
1        control                0.692463
         time                   0.622711
         stochastic             0.527472
         state                  0.480249
         discrete               0.438263
         optimal                0.434315
         dynamic                0.433260
         systems                0.422429
         processes              0.402646
         adaptive               0.398104
2        recognition            0.656475
         pattern                0.593984
         detection              0.593272
         image                  0.587403
         images                 0.540667
         dimensional            0.532123
         classification         0.520903
         filters                0.504324
         speech                 0.498907
         using                  0.464354
3        graphs                 0.772053
         sup                    0.740108
         number                 0.720778
         graph                  0.701702
         sub                    0.701551
         cycles                 0.695806
         trees                  0.673767
         sets                   0.664125
         groups                 0.622460
         designs                0.594136
4        database               0.685566
         retrieval              0.681397
         data                   0.671188
         management             0.582460
         information            0.549630
         base                   0.541097
         storage                0.526647
         knowledge              0.463351
         distributed            0.434420
         performance            0.432143
5        microprocessor         0.544468
         interactive            0.533201
         software               0.525627
         simulation             0.524618
         graphics               0.521364
         design                 0.490170
         processor              0.470266
         machine                0.459903
         interface              0.454628
         robot                  0.453337
6        languages              0.732819
         logic                  0.709611
         theory                 0.602000
         automata               0.595470
         context                0.552895
         association            0.545262
         symbolic               0.544884
         semantics              0.542442
         proof                  0.534679
         free                   0.528657
7        der                    0.608225
         und                    0.558790
         uuml                   0.550439
         von                    0.529574
         des                    0.506531
         zur                    0.475109
         eacute                 0.471331
         ein                    0.470777
         ouml                   0.469876
         auml                   0.460578
8        ritz                   1.003499
         inhaltsadressierbares  0.943386
         codebooks              0.941278
         speichersystem         0.928322
         vorbereitung           0.924116
         integralgleichung      0.920326
         details                0.910183
         relationnelles         0.898930
         descriptor             0.898429
         cassette               0.897780
9        book                   0.817972
         science                0.719414
         review                 0.709262
         scientific             0.700615
         conference             0.700450
         journal                0.602907
         chess   

In [ ]:
# Visualize 10 topics returned by CTM for Preprocess 1
vis_pipeline(ctm=ctm1_10, tp=tp1_10, training_dataset=training_dataset1)

100%|██████████| 625/625 [00:11<00:00, 55.02it/s]


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
7      0.000310  0.000029       1        1  12.422656
2      0.001146 -0.000106       2        1  11.523541
4      0.000949  0.000528       3        1  11.419478
1      0.001348  0.000013       4        1  10.792585
0      0.001027 -0.000546       5        1  10.041581
3     -0.000163 -0.000987       6        1   9.473118
5      0.000452  0.000606       7        1   9.471154
9     -0.000548  0.000765       8        1   9.314342
6     -0.000298 -0.000287       9        1   8.556210
8     -0.004221 -0.000013      10        1   6.985336, topic_info=                                 Term       Freq      Total Category  logprob  \
15309                        problems  11.000000  11.000000  Default  30.0000   
16992                            ritz  10.000000  10.000000  Default  29.0000   
10715                       languages  11.000000  11.000000  Default  28.0000   
2300                             book  11.000000  11.000000  Default  27.0000   
11222                           logic  11.000000  11.000000  Default  26.0000   
3936                          control  11.000000  11.000000  Default  25.0000   
3305                        codebooks  10.000000  10.000000  Default  24.0000   
8145                           graphs  11.000000  11.000000  Default  23.0000   
9434            inhaltsadressierbares  10.000000  10.000000  Default  22.0000   
19247                             sup  11.000000  11.000000  Default  21.0000   
4840                       descriptor   9.000000   9.000000  Default  20.0000   
15298                         problem  11.000000  11.000000  Default  19.0000   
20980                     utilisation   9.000000   9.000000  Default  18.0000   
13583                       numerical  11.000000  11.000000  Default  17.0000   
6320                        equations  11.000000  11.000000  Default  16.0000   
18498                  speichersystem  10.000000  10.000000  Default  15.0000   
2752                         cassette  10.000000  10.000000  Default  14.0000   
14278                       passenger  10.000000  10.000000  Default  13.0000   
18328                        solution  11.000000  11.000000  Default  12.0000   
4867                          details  10.000000  10.000000  Default  11.0000   
13801                             ops  10.000000  10.000000  Default  10.0000   
13573                          number  11.000000  11.000000  Default   9.0000   
8118                            graph  11.000000  11.000000  Default   8.0000   
9567                integralgleichung  10.000000  10.000000  Default   7.0000   
178                         acquiring  10.000000  10.000000  Default   6.0000   
4362                           cycles  10.000000  10.000000  Default   5.0000   
20019                            time  11.000000  11.000000  Default   4.0000   
19209                     suggestions   9.000000   9.000000  Default   3.0000   
21103                          vendor  10.000000  10.000000  Default   2.0000   
16578                  relationnelles  10.000000  10.000000  Default   1.0000   
4802                              der   2.849840  11.674929   Topic1  -9.2225   
20711                             und   2.712385  11.421486   Topic1  -9.2720   
21445                             von   2.634286  11.146674   Topic1  -9.3012   
20989                            uuml   2.689827  11.447932   Topic1  -9.2803   
14002                            ouml   2.481626  10.672981   Topic1  -9.3609   
4823                              des   2.574279  11.353829   Topic1  -9.3242   
5614                           eacute   2.485240  10.962288   Topic1  -9.3594   
22151                             zur   2.494646  11.238322   Topic1  -9.3556   
5813                              ein   2.483863  11.246443   Topic1  -9.3600   
12278                             mit   2.369803  10.750639   Topic1 

---

In [ ]:
# Preprocess 2
preprocessed_docs2 = [' '.join(title) for title in preprocess2(titles=titles_before_1990)]
unpreprocessed_docs2 = titles_before_1990.copy()

In [ ]:
# Perform CTM with num_ctm_topics = 5 for Preprocess 2
ctm2_5, tp2_5, training_dataset2 = ctm_pipeline(preprocessed_docs=preprocessed_docs2,
                                                unpreprocessed_docs=unpreprocessed_docs2,
                                                n_components=5)

Preparing the data...


Batches:   0%|          | 0/200 [00:00<?, ?it/s]

Training the model...


Epoch: [10/10]	 Seen Samples: [400000/400000]	Train Loss: 56.07969094238281	Time: 0:00:14.009648: : 10it [02:17, 13.74s/it]
100%|██████████| 625/625 [00:11<00:00, 56.39it/s]


Calculating Coherence Score...

Coherence Score: 0.49990469269188187


score
topic_id word                         
0        protocol             0.507709
         interface            0.469338
         architecture         0.465486
         storage              0.446741
         information          0.442553
         retrieval            0.439000
         base                 0.434594
         development          0.430663
         database             0.426289
         computer             0.423026
1        von                  0.493741
         uuml                 0.471582
         der                  0.465964
         ein                  0.450150
         auml                 0.433782
         zur                  0.433295
         mit                  0.419098
         ber                  0.413700
         ouml                 0.413487
         die                  0.411087
2        filter               0.471776
         estimation           0.457607
         recognition          0.441834
         pattern              0.439432
         adaptive             0.428347
         image                0.422687
         state                0.413777
         detection            0.410079
         differential         0.401297
         method               0.399754
3        graph                0.524016
         automaton            0.506257
         shortest             0.492552
         tree                 0.486985
         connected            0.470657
         extension            0.463345
         number               0.457046
         minimal              0.456341
         theorem              0.450603
         cycle                0.446930
4        shooting             1.058385
         verschnittproblem    0.864502
         sen                  0.848973
         stamped              0.843550
         hochleistungs        0.842116
         hilbertraum          0.837352
         intervallfunktionen  0.821800
         certaines            0.821356
         projektplanung       0.814688
         diskretisierung      0.813860

In [ ]:
# Visualize 5 topics returned by CTM for Preprocess 2
vis_pipeline(ctm=ctm2_5, tp=tp2_5, training_dataset=training_dataset2)

100%|██████████| 625/625 [00:10<00:00, 56.86it/s]


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2      0.001592  0.000083       1        1  24.411994
0      0.000599 -0.000716       2        1  21.434086
3      0.000377  0.000810       3        1  20.036293
1      0.000347 -0.000161       4        1  20.020399
4     -0.002915 -0.000016       5        1  14.097229, topic_info=                           Term       Freq      Total Category  logprob  \
16642                  shooting  11.000000  11.000000  Default  30.0000   
7988                hilbertraum  11.000000  11.000000  Default  29.0000   
16482                       sen  11.000000  11.000000  Default  28.0000   
19732         verschnittproblem  11.000000  11.000000  Default  27.0000   
8049              hochleistungs  12.000000  12.000000  Default  26.0000   
4823            diskretisierung  11.000000  11.000000  Default  25.0000   
1271                       auto  10.000000  10.000000  Default  24.0000   
9049        intervallfunktionen  11.000000  11.000000  Default  23.0000   
2669                  certaines  11.000000  11.000000  Default  22.0000   
17388                   stamped  11.000000  11.000000  Default  21.0000   
14413            projektplanung  11.000000  11.000000  Default  20.0000   
13239                     param  11.000000  11.000000  Default  19.0000   
9685                komponenten  11.000000  11.000000  Default  18.0000   
19937                       von  12.000000  12.000000  Default  17.0000   
6705                     folgen  11.000000  11.000000  Default  16.0000   
17207               spezialfall  11.000000  11.000000  Default  15.0000   
15440                   relativ  11.000000  11.000000  Default  14.0000   
18080             syntaktischer  11.000000  11.000000  Default  13.0000   
4159   datenverarbeitungsanlage  11.000000  11.000000  Default  12.0000   
4622               differencing  11.000000  11.000000  Default  11.0000   
1369                        axe  11.000000  11.000000  Default  10.0000   
19507                      uuml  12.000000  12.000000  Default   9.0000   
14780                   quellen  11.000000  11.000000  Default   8.0000   
5294         effizienzvergleich  11.000000  11.000000  Default   7.0000   
464                 allgemeines  11.000000  11.000000  Default   6.0000   
4416                        der  12.000000  12.000000  Default   5.0000   
15152                  rechners  11.000000  11.000000  Default   4.0000   
10345                    listen  11.000000  11.000000  Default   3.0000   
15024                   readout  11.000000  11.000000  Default   2.0000   
15800             risikoanalyse  11.000000  11.000000  Default   1.0000   
6545                     filter   5.155477  12.689851   Topic1  -9.3166   
15183               recognition   5.003400  12.422109   Topic1  -9.3466   
13361                   pattern   4.991398  12.445643   Topic1  -9.3490   
5970                 estimation   5.082945  12.697059   Topic1  -9.3308   
14092                prediction   4.527422  11.439588   Topic1  -9.4466   
204                    adaptive   4.936371  12.488191   Topic1  -9.3601   
8339             identification   4.657125  11.834885   Topic1  -9.4183   
4627               differential   4.804636  12.225941   Topic1  -9.3871   
4477                  detection   4.847012  12.407993   Topic1  -9.3783   
8399                      image   4.908513  12.598619   Topic1  -9.3657   
12508                 nonlinear   4.720742  12.155761   Topic1  -9.4047   
17428                     state   4.864970  12.584661   Topic1  -9.3746   
18603                      time   4.787319  12.541792   Topic1  -9.3907   
13241                 parameter   4.580614  12.014469   Topic1  -9.4349   
4657                    digital   4.632860  12.209570   Topic1  -9.4235   
6901                  frequency   4.383770  11.555907   Topic1  -9.4788   
17552                stochastic   4.646020  12.262412 

In [ ]:
# Perform CTM with num_ctm_topics > 5 for Preprocess 2
ctm2_10, tp2_10, training_dataset2 = ctm_pipeline(preprocessed_docs=preprocessed_docs2,
                                                  unpreprocessed_docs=unpreprocessed_docs2,
                                                  n_components=10)

Preparing the data...


Batches:   0%|          | 0/200 [00:00<?, ?it/s]

Training the model...


Epoch: [10/10]	 Seen Samples: [400000/400000]	Train Loss: 56.53677713623047	Time: 0:00:14.151149: : 10it [02:17, 13.73s/it]
100%|██████████| 625/625 [00:10<00:00, 62.10it/s]


Calculating Coherence Score...

Coherence Score: 0.5155490262053901


score
topic_id word                    
0        computer        0.738344
         software        0.728974
         science         0.710122
         engineering     0.692929
         research        0.589636
         microprocessor  0.586191
         graphic         0.575176
         development     0.569168
         microcomputer   0.496912
         technology      0.489287
1        cardinal        0.751456
         identity        0.750657
         definability    0.724449
         algebra         0.716949
         calculus        0.673955
         quantifier      0.673896
         ordinal         0.665496
         truth           0.655259
         independence    0.649938
         axiom           0.639082
2        equation        0.673819
         method          0.584644
         solution        0.583966
         function        0.493302
         numerical       0.457754
         problem         0.450924
         error           0.434649
         approximation   0.429368
         nonlinear       0.423499
         matrix          0.419324
3        recognition     0.780604
         image           0.756284
         pattern         0.668400
         speech          0.588916
         dimensional     0.499570
         feature         0.498377
         digital         0.480365
         using           0.472331
         line            0.470791
         classification  0.450864
4        tree            0.741840
         graph           0.732645
         sup             0.643389
         sub             0.619384
         complexity      0.566831
         path            0.562655
         finding         0.556507
         minimal         0.518588
         number          0.507041
         convex          0.500846
5        language        0.626610
         program         0.557606
         grammar         0.556328
         translation     0.539931
         context         0.509654
         test            0.508848
         free            0.502136
         fiber           0.493530
         magnetic        0.486303
         verification    0.464632
6        association     0.905584
         transit         0.845281
         timed           0.833754
         symbolic        0.832716
         matiques        0.825180
         meeting         0.823398
         hierarchically  0.821423
         kleine          0.820060
         occurring       0.813281
         cartography     0.812848
7        database        0.773867
         information     0.688024
         data            0.663839
         retrieval       0.609350
         base            0.577783
         knowledge       0.545834
         storage         0.487429
         management      0.442345
         relational      0.431255
         network         0.420760
8        control         0.789602
         time            0.521323
         model           0.496832
         dynamic         0.474065
         stochastic      0.433860
         simulation      0.430701
         optimal         0.429047
         state           0.418688
         process         0.395876
         robot           0.382641
9        de              0.610409
         uuml            0.559952
         von             0.555103
         und             0.544395
         der             0.531029
         auml            0.491072
         zur             0.468764
         die             0.442266
         ouml            0.426917
         mit             0.423874

In [ ]:
# Visualize 10 topics returned by CTM for Preprocess 2
vis_pipeline(ctm=ctm2_10, tp=tp2_10, training_dataset=training_dataset2)

100%|██████████| 625/625 [00:10<00:00, 58.97it/s]


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
9     -0.000396 -0.000196       1        1  12.316199
2     -0.001442  0.000656       2        1  11.020800
3     -0.001052  0.000004       3        1  11.003330
0     -0.000170 -0.000872       4        1  10.960287
5     -0.000289 -0.000115       5        1   9.969443
7     -0.000608 -0.000522       6        1   9.742479
8     -0.001195 -0.000221       7        1   9.511809
4     -0.000324  0.000659       8        1   9.286354
1      0.001599  0.000891       9        1   8.374955
6      0.003877 -0.000284      10        1   7.814345, topic_info=                    Term       Freq      Total Category  logprob  loglift
3617             control  12.000000  12.000000  Default  30.0000  30.0000
4079            database  12.000000  12.000000  Default  29.0000  29.0000
18909               tree  12.000000  12.000000  Default  28.0000  28.0000
15183        recognition  12.000000  12.000000  Default  27.0000  27.0000
7499               graph  12.000000  12.000000  Default  26.0000  26.0000
8696         information  12.000000  12.000000  Default  25.0000  25.0000
8399               image  12.000000  12.000000  Default  24.0000  24.0000
4078                data  12.000000  12.000000  Default  23.0000  23.0000
5821            equation  12.000000  12.000000  Default  22.0000  22.0000
15680          retrieval  12.000000  12.000000  Default  21.0000  21.0000
1490                base  11.000000  11.000000  Default  20.0000  20.0000
7962      hierarchically  10.000000  10.000000  Default  19.0000  19.0000
17880                sup  11.000000  11.000000  Default  18.0000  18.0000
3363            computer  12.000000  12.000000  Default  17.0000  17.0000
18604              timed  10.000000  10.000000  Default  16.0000  16.0000
18822            transit  11.000000  11.000000  Default  15.0000  15.0000
15196     reconfigurable  10.000000  10.000000  Default  14.0000  14.0000
19063            turbine  10.000000  10.000000  Default  13.0000  13.0000
13361            pattern  12.000000  12.000000  Default  12.0000  12.0000
17001           software  12.000000  12.000000  Default  11.0000  11.0000
3320          complexity  11.000000  11.000000  Default  10.0000  10.0000
18603               time  12.000000  12.000000  Default   9.0000   9.0000
11473              model  12.000000  12.000000  Default   8.0000   8.0000
5657         engineering  12.000000  12.000000  Default   7.0000   7.0000
13352               path  11.000000  11.000000  Default   6.0000   6.0000
16277            science  12.000000  12.000000  Default   5.0000   5.0000
17695                sub  12.000000  12.000000  Default   4.0000   4.0000
9616           knowledge  11.000000  11.000000  Default   3.0000   3.0000
15874           rollback  10.000000  10.000000  Default   2.0000   2.0000
17037           solution  11.000000  11.000000  Default   1.0000   1.0000
4200                  de   3.057191  12.511455   Topic1  -9.1550   0.6851
19937                von   2.892702  12.045343   Topic1  -9.2104   0.6678
19507               uuml   2.906764  12.342690   Topic1  -9.2055   0.6482
19240                und   2.861892  12.282396   Topic1  -9.2211   0.6376
4416                 der   2.823895  12.225817   Topic1  -9.2344   0.6288
1189                auml   2.713283  12.009888   Topic1  -9.2744   0.6067
20624                zur   2.653428  11.839409   Topic1  -9.2967   0.5987
11431                mit   2.536948  11.412045   Topic1  -9.3416   0.5905
13084               ouml   2.544680  11.572284   Topic1  -9.3385   0.5796
4605                 die   2.584040  11.827188   Topic1  -9.3232   0.5732
5346                 ein   2.496239  11.526649   Topic1  -9.3578   0.5644
5162              eacute   2.484401  11.591031   Topic1  -9.3625   0.5541
1699                 ber   2.391733  11.242443   Topic1  -9.4005   0.5466
4029                  da   2.450221  11.524813   Topi

### From 1990 to 2009

Add your code for topic modelling the period from 1990 to 2009 here - similar to what you did for before 1990s

In [ ]:
# Preprocess 1
preprocessed_docs1 = [' '.join(title) for title in preprocess1(titles=titles_from_1990_to_2009)]
unpreprocessed_docs1 = titles_from_1990_to_2009.copy()

In [ ]:
# Perform CTM with num_ctm_topics = 5 for Preprocess 1
ctm1_5, tp1_5, training_dataset1 = ctm_pipeline(preprocessed_docs=preprocessed_docs1,
                                                unpreprocessed_docs=unpreprocessed_docs1,
                                                n_components=5,
                                                num_epochs=5)

Preparing the data...


Batches:   0%|          | 0/1218 [00:00<?, ?it/s]

Training the model...


Epoch: [5/5]	 Seen Samples: [1217600/1217905]	Train Loss: 72.7546156804289	Time: 0:02:53.834974: : 5it [14:00, 168.05s/it]
100%|██████████| 3806/3806 [02:04<00:00, 30.55it/s]


Calculating Coherence Score...

Coherence Score: 0.41378187173185


score
topic_id word                    
0        fault           0.928511
         networks        0.899160
         processor       0.871377
         manipulators    0.863587
         robot           0.853769
         hoc             0.851420
         power           0.847965
         traffic         0.834675
         routing         0.825656
         wireless        0.820710
1        endoscopic      1.758963
         adaptively      1.744331
         modulator       1.721121
         asr             1.640426
         coaxial         1.640358
         adjustable      1.621933
         hybridization   1.604924
         erlang          1.599565
         trains          1.596471
         timestamp       1.591483
2        classification  0.885037
         segmentation    0.865532
         feature         0.850461
         magnetic        0.847775
         imaging         0.846300
         temporal        0.836905
         extraction      0.825393
         brain           0.817709
         resolution      0.806536
         spatial         0.797589
3        integral        1.035388
         quasi           1.002353
         convex          0.962093
         symmetric       0.943246
         elements        0.942950
         one             0.936541
         interior        0.915814
         polynomial      0.898809
         newton          0.896684
         equation        0.887804
4        exploring       1.094506
         electronic      1.028355
         interfaces      1.007038
         collaboration   0.992015
         users           0.989699
         experience      0.966075
         journal         0.964767
         literature      0.959216
         social          0.944446
         conceptual      0.944353

In [ ]:
# Perform CTM with num_ctm_topics > 5 for Preprocess 1
ctm1_10, tp1_10, training_dataset1 = ctm_pipeline(preprocessed_docs=preprocessed_docs1,
                                                  unpreprocessed_docs=unpreprocessed_docs1,
                                                  n_components=10,
                                                  num_epochs=5)

Preparing the data...


Batches:   0%|          | 0/1218 [00:00<?, ?it/s]

Training the model...


Epoch: [5/5]	 Seen Samples: [1217600/1217905]	Train Loss: 73.10362506019301	Time: 0:02:49.093551: : 5it [14:16, 171.34s/it]
100%|██████████| 3806/3806 [02:14<00:00, 28.37it/s]


Calculating Coherence Score...

Coherence Score: 0.6059805506684288


score
topic_id word                    
0        von             1.214168
         der             1.155986
         mit             1.149544
         die             1.146298
         und             1.124431
         uuml            1.095200
         ein             1.067482
         auml            1.060123
         ber             1.058679
         das             1.049396
1        trees           1.807695
         graph           1.731702
         groups          1.720730
         sets            1.635881
         cycles          1.591891
         complete        1.583040
         regular         1.571843
         number          1.556689
         numbers         1.550949
         classes         1.535277
2        functional      1.296427
         brain           1.239591
         protein         1.160618
         response        1.117112
         imaging         1.093423
         magnetic        1.064104
         molecular       1.049582
         effect          0.985416
         fmri            0.982575
         activity        0.968683
3        mining          1.174242
         law             1.121918
         databases       1.107560
         scientific      1.060295
         library         1.051100
         electronic      1.047549
         special         1.027224
         journal         1.024620
         eacute          0.998076
         world           0.983283
4        stabilisation   2.424193
         rotor           2.224606
         memoryless      2.171303
         paced           2.158421
         nonaffine       2.148460
         asr             2.140586
         separate        2.134026
         targeted        2.100764
         characterized   2.083865
         amplitudes      2.056555
5        wireless        1.471525
         routing         1.368168
         protocol        1.300000
         allocation      1.298964
         atm             1.249810
         traffic         1.208594
         processor       1.172879
         performance     1.146550
         protocols       1.123541
         hoc             1.095255
6        control         1.403559
         feedback        1.250874
         fuzzy           1.212217
         robot           1.156573
         controller      0.950490
         state           0.913883
         neural          0.909356
         uncertain       0.909077
         delay           0.901651
         systems         0.901478
7        image           1.396116
         classification  1.338629
         color           1.261674
         blind           1.197921
         coding          1.194820
         compression     1.190161
         recognition     1.189846
         wavelet         1.179290
         feature         1.147054
         images          1.128711
8        oriented        1.358723
         virtual         1.339961
         development     1.311339
         software        1.254496
         process         1.246143
         engineering     1.184863
         knowledge       1.174045
         specification   1.107034
         interactive     1.077065
         support         1.049583
9        solution        1.489621
         problems        1.364005
         equation        1.326905
         convergence     1.274527
         element         1.243830
         equations       1.222693
         solving         1.221846
         solutions       1.204402
         methods         1.175059
         numerical       1.172112

---

In [ ]:
# Preprocess 2
preprocessed_docs2 = [' '.join(title) for title in preprocess2(titles=titles_from_1990_to_2009)]
unpreprocessed_docs2 = titles_from_1990_to_2009.copy()

In [ ]:
# Perform CTM with num_ctm_topics = 5 for Preprocess 2
ctm2_5, tp2_5, training_dataset2 = ctm_pipeline(preprocessed_docs=preprocessed_docs2,
                                                unpreprocessed_docs=unpreprocessed_docs2,
                                                n_components=5,
                                                num_epochs=5)

Preparing the data...


Batches:   0%|          | 0/1218 [00:00<?, ?it/s]

Training the model...


Epoch: [5/5]	 Seen Samples: [1217600/1217905]	Train Loss: 72.93969926169918	Time: 0:02:37.465551: : 5it [13:19, 159.87s/it]
100%|██████████| 3806/3806 [02:05<00:00, 30.31it/s]


Calculating Coherence Score...

Coherence Score: 0.47531015153975364


score
topic_id word                    
0        feature         0.873035
         speech          0.863746
         magnetic        0.858771
         image           0.853624
         classification  0.825752
         protein         0.815846
         recognition     0.808509
         imaging         0.806868
         clustering      0.802464
         segmentation    0.799907
1        control         0.923428
         fault           0.904509
         controller      0.887709
         traffic         0.880375
         allocation      0.877466
         manipulator     0.864316
         network         0.855748
         power           0.826303
         atm             0.818239
         delay           0.805425
2        organizational  1.052692
         managing        1.024752
         enterprise      1.020020
         health          1.010891
         project         0.987880
         practice        0.978967
         strategic       0.968854
         conceptual      0.968077
         towards         0.938888
         course          0.934997
3        convex          0.977871
         newton          0.970639
         matrix          0.928925
         approximation   0.918825
         integral        0.905037
         sup             0.899676
         second          0.897059
         regular         0.893386
         eigenvalue      0.879733
         problem         0.879055
4        diffusivity     1.833038
         rician          1.741874
         multiband       1.696050
         adjustable      1.693823
         mosfet          1.690912
         eigen           1.667888
         noncontact      1.666320
         reinforced      1.641813
         lsi             1.635926
         deblocking      1.630001

In [ ]:
# Perform CTM with num_ctm_topics > 5 for Preprocess 2
ctm2_10, tp2_10, training_dataset2 = ctm_pipeline(preprocessed_docs=preprocessed_docs2,
                                                  unpreprocessed_docs=unpreprocessed_docs2,
                                                  n_components=10,
                                                  num_epochs=5)

Preparing the data...


Batches:   0%|          | 0/1218 [00:00<?, ?it/s]

Training the model...


Epoch: [5/5]	 Seen Samples: [1217600/1217905]	Train Loss: 73.18922590939977	Time: 0:02:34.340076: : 5it [12:43, 152.79s/it]
100%|██████████| 3806/3806 [02:03<00:00, 30.82it/s]


Calculating Coherence Score...

Coherence Score: 0.5871359403479495


score
topic_id word                   
0        brain          1.232073
         measurement    1.135414
         imaging        1.057009
         response       1.053084
         fmri           1.003682
         gene           0.955003
         temporal       0.946933
         activation     0.935139
         magnetic       0.931486
         correlation    0.876574
1        image          1.390675
         recognition    1.380452
         color          1.363393
         feature        1.336062
         extraction     1.208263
         compression    1.204802
         face           1.181742
         texture        1.181073
         shape          1.177049
         segmentation   1.121058
2        regular        1.866531
         complete       1.754488
         number         1.661542
         ordering       1.605984
         ordered        1.576658
         family         1.566170
         cycle          1.555003
         algebra        1.549531
         permutation    1.545304
         covering       1.539049
3        select         2.383633
         mse            2.300750
         tank           2.279467
         footprint      2.276819
         timer          2.263479
         void           2.224574
         localizing     2.219119
         enlargement    2.201250
         duffing        2.192724
         synapsis       2.171408
4        oriented       1.600938
         specification  1.535505
         language       1.360880
         formal         1.350383
         embedded       1.341202
         verification   1.284572
         concurrent     1.220222
         relational     1.203530
         database       1.184616
         program        1.157906
5        special        1.456921
         der            1.359035
         guest          1.356952
         und            1.286894
         von            1.242541
         uuml           1.217982
         introduction   1.210260
         auml           1.198964
         mit            1.163843
         de             1.150858
6        numerical      1.312137
         equation       1.273853
         solution       1.270214
         element        1.235698
         flow           1.148870
         convergence    1.108126
         boundary       1.094641
         method         1.050118
         problem        1.043032
         solving        1.012175
7        control        1.409570
         robot          1.223190
         feedback       1.177452
         fuzzy          1.097582
         controller     1.000378
         output         0.953510
         state          0.936165
         manipulator    0.917500
         stability      0.874432
         time           0.874325
8        online         1.507120
         social         1.379360
         health         1.319128
         community      1.309875
         education      1.291065
         commerce       1.277083
         electronic     1.266526
         research       1.249117
         student        1.245532
         perspective    1.245372
9        wireless       1.275820
         channel        1.270645
         traffic        1.258350
         capacity       1.169144
         routing        1.145884
         transmission   1.119833
         atm            1.093780
         ad             1.088434
         protocol       1.082274
         network        1.047817

### From 2010 onwards



Add your code for topic modelling the period from 2010 onwards - similar to what you did for before 1990s

In [ ]:
# Preprocess 1
preprocessed_docs1 = [' '.join(title) for title in preprocess1(titles=titles_from_2010)]
unpreprocessed_docs1 = titles_from_2010.copy()

In [ ]:
# Perform CTM with num_ctm_topics = 5 for Preprocess 1
ctm1_5, tp1_5, training_dataset1 = ctm_pipeline(preprocessed_docs=preprocessed_docs1,
                                                unpreprocessed_docs=unpreprocessed_docs1,
                                                n_components=5,
                                                num_epochs=1)

Preparing the data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/2912 [00:00<?, ?it/s]

Training the model...


Epoch: [1/1]	 Seen Samples: [582336/582378]	Train Loss: 96.50312099169605	Time: 0:10:30.064692: : 1it [10:30, 630.07s/it]
100%|██████████| 9100/9100 [07:14<00:00, 20.97it/s]


Calculating Coherence Score...

Coherence Score: 0.4619359833852517


score
topic_id word                    
0        embedded        0.779345
         chip            0.683319
         secure          0.644839
         latency         0.635115
         enabled         0.599576
         networks        0.598849
         radio           0.597976
         encryption      0.593687
         cellular        0.592446
         attack          0.591226
1        systematic      0.753501
         exploratory     0.730743
         research        0.707078
         effectiveness   0.651241
         among           0.649810
         student         0.647876
         citation        0.644531
         review          0.631440
         role            0.627295
         perspective     0.625353
2        stability       0.679144
         numerical       0.653347
         stochastic      0.639403
         output          0.624204
         stabilization   0.596218
         linear          0.588995
         actuator        0.587313
         lagrangian      0.586831
         formulation     0.580289
         dependent       0.579002
3        hyperspectral   0.667092
         similarity      0.635181
         pattern         0.606977
         features        0.603608
         fusion          0.598854
         object          0.590919
         imaging         0.587422
         recognition     0.584882
         identification  0.584203
         super           0.580537
4        honk            1.204305
         inclinations    1.123731
         wilt            1.045366
         chem            0.980838
         crustal         0.964609
         enso            0.959150
         gedi            0.958404
         caption         0.943543
         vii             0.940037
         eagle           0.916479

In [ ]:
# Perform CTM with num_ctm_topics > 5 for Preprocess 1
ctm1_10, tp1_10, training_dataset1 = ctm_pipeline(preprocessed_docs=preprocessed_docs1,
                                                  unpreprocessed_docs=unpreprocessed_docs1,
                                                  n_components=10,
                                                  num_epochs=1)

Preparing the data...


Batches:   0%|          | 0/2912 [00:00<?, ?it/s]

Training the model...


Epoch: [1/1]	 Seen Samples: [582336/582378]	Train Loss: 96.67653725181049	Time: 0:10:32.403489: : 1it [10:32, 632.41s/it]
100%|██████████| 9100/9100 [07:20<00:00, 20.64it/s]


Calculating Coherence Score...

Coherence Score: 0.6048283553539717


score
topic_id word                      
0        filter            0.763369
         transform         0.755963
         noise             0.744380
         reconstruction    0.715088
         frequency         0.653835
         wavelet           0.653624
         estimation        0.650377
         fast              0.628806
         enhancement       0.598028
         sparse            0.597241
1        deep              0.850866
         recognition       0.826687
         supervised        0.820277
         adversarial       0.814510
         convolutional     0.814370
         learning          0.807022
         segmentation      0.775645
         semantic          0.762553
         attention         0.738956
         face              0.724818
2        moth              1.119338
         benthic           1.068504
         multiparametric   1.045208
         furnace           1.038311
         roadside          1.036407
         braking           1.029698
         spatiotemporally  1.028214
         equators          1.014920
         memoryless        1.007991
         wavefront         1.003911
3        stability         0.954882
         nonlinear         0.929442
         equation          0.915776
         differential      0.905428
         stochastic        0.900016
         equations         0.861895
         second            0.853506
         order             0.849240
         boundary          0.843418
         fractional        0.835100
4        brain             0.876633
         gene              0.843943
         functional        0.793899
         connectivity      0.780851
         fmri              0.767817
         protein           0.735487
         activity          0.700923
         human             0.664901
         cortex            0.653222
         resonance         0.636477
5        planning          0.954159
         chain             0.880674
         manufacturing     0.851046
         supply            0.833228
         robots            0.803849
         maintenance       0.781544
         decision          0.734609
         safety            0.732398
         process           0.721145
         autonomous        0.682889
6        remote            0.780696
         urban             0.747095
         mapping           0.733910
         water             0.703630
         surface           0.697998
         vegetation        0.688817
         satellite         0.676929
         sea               0.674729
         sentinel          0.674223
         observations      0.668797
7        complete          1.056855
         lattices          1.039352
         classes           1.018242
         prime             1.013573
         regular           1.003397
         primitive         1.001459
         numbers           1.000330
         categories        0.997328
         proof             0.992879
         graphs            0.970443
8        research          0.937737
         social            0.936782
         online            0.896107
         innovation        0.871345
         media             0.867026
         academic          0.866506
         education         0.860041
         open              0.815474
         science           0.811944
         communities       0.807112
9        wireless          1.042435
         communications    1.025698
         transmission      0.925485
         radio             0.876273
         routing           0.837601
         sensor            0.818510
         protocol          0.809614
         energy            0.805756
         networks          0.785902
         massive           0.785227

---

In [ ]:
# Preprocess 2
preprocessed_docs2 = [' '.join(title) for title in preprocess2(titles=titles_from_2010)]
unpreprocessed_docs2 = titles_from_2010.copy()

In [ ]:
# Perform CTM with num_ctm_topics = 5 for Preprocess 2
ctm2_5, tp2_5, training_dataset2 = ctm_pipeline(preprocessed_docs=preprocessed_docs2,
                                                unpreprocessed_docs=unpreprocessed_docs2,
                                                n_components=5,
                                                num_epochs=1)

Preparing the data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/2912 [00:00<?, ?it/s]

Training the model...


Epoch: [1/1]	 Seen Samples: [582336/582378]	Train Loss: 96.8119128367995	Time: 0:10:00.113361: : 1it [10:00, 600.12s/it]
100%|██████████| 9100/9100 [06:57<00:00, 21.79it/s]


Calculating Coherence Score...

Coherence Score: 0.4192618079841817


score
topic_id word                      
0        adversarial       0.718014
         object            0.654374
         classification    0.638774
         unsupervised      0.603797
         supervised        0.597909
         generative        0.594292
         feature           0.589436
         convolutional     0.587992
         cnn               0.587959
         denoising         0.579962
1        numerical         0.657034
         equation          0.631584
         discontinuous     0.615324
         element           0.612707
         landsat           0.601389
         fluid             0.597199
         sentinel          0.582829
         temperature       0.581978
         stokes            0.580463
         measurement       0.572483
2        transmission      0.637069
         tolerant          0.621914
         radio             0.605333
         routing           0.601811
         consensus         0.598133
         control           0.597963
         feedback          0.595011
         sliding           0.589284
         switching         0.584538
         power             0.583718
3        mussel            1.182206
         pixjs             1.131030
         uncalibrated      1.117532
         electromagnetics  1.093853
         hangzhou          1.070186
         transhumanists    1.069873
         nanosecond        1.068135
         elmer             1.058227
         simvodis          1.052227
         infarction        1.047330
4        usage             0.716352
         engineering       0.697650
         innovative        0.683454
         digital           0.670077
         requirement       0.668419
         intention         0.642809
         opportunity       0.641396
         exploratory       0.623776
         need              0.622677
         consumer          0.614363

In [ ]:
# Perform CTM with num_ctm_topics > 5 for Preprocess 2
ctm2_10, tp2_10, training_dataset2 = ctm_pipeline(preprocessed_docs=preprocessed_docs2,
                                                  unpreprocessed_docs=unpreprocessed_docs2,
                                                  n_components=10,
                                                  num_epochs=1)

Preparing the data...


Batches:   0%|          | 0/2912 [00:00<?, ?it/s]

Training the model...


Epoch: [1/1]	 Seen Samples: [582336/582378]	Train Loss: 97.01362597285974	Time: 0:10:01.912550: : 1it [10:01, 601.92s/it]
100%|██████████| 9100/9100 [07:05<00:00, 21.37it/s]


Calculating Coherence Score...

Coherence Score: 0.6201285836776325


score
topic_id word                      
0        imaging           0.852295
         magnetic          0.801694
         fmri              0.797483
         functional        0.789960
         resonance         0.787509
         brain             0.772027
         mri               0.720281
         connectivity      0.704002
         reconstruction    0.698404
         cortex            0.681966
1        urban             0.818661
         forest            0.753368
         satellite         0.737614
         mapping           0.722854
         area              0.708108
         remote            0.702152
         water             0.682818
         change            0.675109
         observation       0.669726
         temperature       0.629948
2        chain             1.043018
         supply            1.011101
         blockchain        0.886962
         smart             0.816908
         security          0.795362
         infrastructure    0.773092
         manufacturing     0.760921
         industrial        0.754896
         thing             0.746290
         cloud             0.731165
3        solving           0.849774
         solution          0.842475
         problem           0.826616
         convergence       0.824496
         matrix            0.821885
         approximation     0.794479
         convex            0.760500
         exact             0.759675
         set               0.758687
         operator          0.752034
4        deep              0.944310
         convolutional     0.865959
         recognition       0.860039
         feature           0.822141
         supervised        0.798095
         generative        0.775388
         classification    0.762414
         attention         0.757049
         classifier        0.749469
         semantic          0.735381
5        wireless          0.922105
         energy            0.850273
         channel           0.842203
         transmission      0.821690
         routing           0.803922
         communication     0.778708
         radio             0.747821
         sensor            0.739435
         cellular          0.731556
         mimo              0.718668
6        control           0.909190
         output            0.763013
         feedback          0.727204
         stability         0.711972
         uncertain         0.697425
         robot             0.688949
         varying           0.685847
         nonlinear         0.683091
         unknown           0.676562
         delay             0.674645
7        social            0.930166
         among             0.873505
         student           0.864927
         medium            0.808483
         role              0.804158
         online            0.780967
         reality           0.774781
         teaching          0.767730
         intention         0.758803
         research          0.751628
8        na                0.957675
         section           0.865709
         egrave            0.859534
         da                0.815623
         th                0.794212
         et                0.781021
         der               0.774149
         biology           0.773861
         le                0.769773
         de                0.754096
9        varied            1.348475
         monitored         1.308079
         rotated           1.175670
         spatiotemporally  1.171461
         bfod              1.160165
         steered           1.147246
         subsea            1.145089
         lagged            1.123444
         jaya              1.116140
         austria           1.110845

📝❓ Again: Assign a name to each topic based on the topic’s top words (for each period). List all topic names in your report.

✅ Assigning name to each generated topic:
## **Before 1990**

| Preprocessing   | Number of Topics | Topic Index | Topic Name                           |
|------------------|------------------|-------------|---------------------------------------|
| Preprocessing 1 | 5                | 0           | **Incoherent**   |
| Preprocessing 1 | 5                | 1           | **Incoherent**     |
| Preprocessing 1 | 5                | 2           | Database Management and Retrieval |
| Preprocessing 1 | 5                | 3           | Signal Processing |
| Preprocessing 1 | 5                | 4           | Graph Theory          |
| Preprocessing 1 | 10               | 0           | Numerical Methods for Differential Equations                      |
| Preprocessing 1 | 10               | 1           | Stochastic and Optimal Control                 |
| Preprocessing 1 | 10               | 2           | Pattern Recognition and Detection               |
| Preprocessing 1 | 10               | 3           | Graph Theory       |
| Preprocessing 1 | 10               | 4           | Databases and Information Retrieval        |
| Preprocessing 1 | 10               | 5           | Microprocessors and Interactive Software                         |
| Preprocessing 1 | 10               | 6           | Formal Languages, Logic and Automata                |
| Preprocessing 1 | 10               | 7           | **Incoherent**                      |
| Preprocessing 1 | 10               | 8           | **Incoherent**                 |
| Preprocessing 1 | 10               | 9           | Scientific Publishing                     |
| Preprocessing 2 | 5                | 0           | Computer Systems Architecture                         |
| Preprocessing 2 | 5                | 1           | **Incoherent**                |
| Preprocessing 2 | 5                | 2           | Pattern Recognition and Detection                 |
| Preprocessing 2 | 5                | 3           | Graph Theory                 |
| Preprocessing 2 | 5                | 4           | **Incoherent**                |
| Preprocessing 2 | 10               | 0           | Software Engineering                      |
| Preprocessing 2 | 10               | 1           | Set Theory                     |
| Preprocessing 2 | 10               | 2           | Numerical Methods               |
| Preprocessing 2 | 10               | 3           | Pattern Recognition and Detection                  |
| Preprocessing 2 | 10               | 4           | Graph Theory                      |
| Preprocessing 2 | 10               | 5           | Programming Languages                  |
| Preprocessing 2 | 10               | 6           | **Incoherent**                  |
| Preprocessing 2 | 10               | 7           | Databases and Information Retrieval               |
| Preprocessing 2 | 10               | 8           | Control Systems and Robotics                |
| Preprocessing 2 | 10               | 9           | **Incoherent**              |
---

## **1990-2009**

| Preprocessing   | Number of Topics | Topic Index | Topic Name                           |
|------------------|------------------|-------------|---------------------------------------|
| Preprocessing 1 | 5                | 0           | Robotics                 |
| Preprocessing 1 | 5                | 1           | **Incoherent**                     |
| Preprocessing 1 | 5                | 2           | ML for Biomedicine                |
| Preprocessing 1 | 5                | 3           | Numerical Methods and Optimization        |
| Preprocessing 1 | 5                | 4           | Scholar Communication               |
| Preprocessing 1 | 10               | 0           | **Incoherent**           |
| Preprocessing 1 | 10               | 1           | Discrete Math           |
| Preprocessing 1 | 10               | 2           | Biomedical Science              |
| Preprocessing 1 | 10               | 3           | Data Mining                      |
| Preprocessing 1 | 10               | 4           | Specialized Control                       |
| Preprocessing 1 | 10               | 5           | Wireless Networking            |
| Preprocessing 1 | 10               | 6           | Robotics                    |
| Preprocessing 1 | 10               | 7           | Image Processing                     |
| Preprocessing 1 | 10               | 8           | Software Engineering                |
| Preprocessing 1 | 10               | 9           | Numerical Methods                   |
| Preprocessing 2 | 5                | 0           | ML for Biomedical Imaging                   |
| Preprocessing 2 | 5                | 1           | Networked Control                    |
| Preprocessing 2 | 5                | 2           | Health Management                |
| Preprocessing 2 | 5                | 3           | Numerical Methods and Optimization          |
| Preprocessing 2 | 5                | 4           | **Incoherent**                         |
| Preprocessing 2 | 10               | 0           | Brain Imaging              |
| Preprocessing 2 | 10               | 1           | Computer Vision                   |
| Preprocessing 2 | 10               | 2           | Algebra                    |
| Preprocessing 2 | 10               | 3           | Computer Modeling                |
| Preprocessing 2 | 10               | 4           | Programming                       |
| Preprocessing 2 | 10               | 5           | **Incoherent**              |
| Preprocessing 2 | 10               | 6           | Numerical methods for PDEs                    |
| Preprocessing 2 | 10               | 7           | Robotics and Control Systems                    |
| Preprocessing 2 | 10               | 8           | Online Platforms                 |
| Preprocessing 2 | 10               | 9           | Wireless Networking              |
---

## **2010 Onwards**

| Preprocessing   | Number of Topics | Topic Index | Topic Name                           |
|------------------|------------------|-------------|---------------------------------------|
| Preprocessing 1 | 5                | 0           | IoT Systems   |
| Preprocessing 1 | 5                | 1           | Research Evaluation            |
| Preprocessing 1 | 5                | 2           | Stochastic Control              |
| Preprocessing 1 | 5                | 3           | Pattern Recognition        |
| Preprocessing 1 | 5                | 4           | **Incoherent**        |
| Preprocessing 1 | 10               | 0           | Signal Processing                      |
| Preprocessing 1 | 10               | 1           | Deep Learning                 |
| Preprocessing 1 | 10               | 2           | **Incoherent**                 |
| Preprocessing 1 | 10               | 3           | Differential Equations              |
| Preprocessing 1 | 10               | 4           | Brain Imaging                       |
| Preprocessing 1 | 10               | 5           | Industrial Robotics                 |
| Preprocessing 1 | 10               | 6           | Environmental Mapping              |
| Preprocessing 1 | 10               | 7           | Number Theory                  |
| Preprocessing 1 | 10               | 8           | Science Impact             |
| Preprocessing 1 | 10               | 9           | Wireless Communications                      |
| Preprocessing 2 | 5                | 0           | Deep Learning   |
| Preprocessing 2 | 5                | 1           | Numerical PDEs            |
| Preprocessing 2 | 5                | 2           | Networked Control              |
| Preprocessing 2 | 5                | 3           | **Incoherent**        |
| Preprocessing 2 | 5                | 4           | User Requirements Engineering        |
| Preprocessing 2 | 10               | 0           | Brain Imaging  |
| Preprocessing 2 | 10               | 1           | Environmental Mapping                      |
| Preprocessing 2 | 10               | 2           | Innovations for Industrial Supply Chains          |
| Preprocessing 2 | 10               | 3           | Convex Optimization                      |
| Preprocessing 2 | 10               | 4           | Deep Learning               |
| Preprocessing 2 | 10               | 5           | Wireless Communications                      |
| Preprocessing 2 | 10               | 6           | Control Systems                |
| Preprocessing 2 | 10               | 7           | Online Teaching                  |
| Preprocessing 2 | 10               | 8           | **Incoherent**                    |
| Preprocessing 2 | 10               | 9           | **Incoherent**        |
---

📝❓ Bianchi et al. 2021 claim that their approach produces more coherent topics than previous methods. Let’s test this claim by comparing the coherence of the topics produced by CTM with the topics produced by LDA. Describe your observations in 3-4 sentences.

✅ Comparison of coherence scores across different configurations for LDA and CTM:
| Period      | Preprocess   | # Topics |       LDA  | CTM  |
| ----------- | ------------ | -------: | ------------------: | ------------: |
| Before 1990 | Preprocess 1 |        5 | 0.388631 | **0.472060** |
|             | Preprocess 1 |       10 | 0.396187 | **0.466865** |
|             | Preprocess 2 |        5 |  0.381805 |**0.499905** |
|             | Preprocess 2 |       10 |  0.364262 |**0.515549**|
| 1990–2009   | Preprocess 1 |        5 |  0.355101 |**0.413782** |
|             | Preprocess 1 |       10 | 0.374973 | **0.605981** |
|             | Preprocess 2 |        5 | 0.341335 |  **0.475310** |
|             | Preprocess 2 |       10 | 0.369782 |  **0.587136** |
| After 2010  | Preprocess 1 |        5 |  0.369827 |**0.461936** |
|             | Preprocess 1 |       10 | 0.311114 | **0.604828** |
|             | Preprocess 2 |        5 | 0.338319|  **0.419262** |
|             | Preprocess 2 |       10 | 0.322945 | **0.620129** |

Across identical experimental settings, **CTM achieves higher coherence scores than LDA in every case**, indicating more coherent topics. Qualitatively, CTM also produces topics whose top words are more conceptually aligned, whereas LDA occasionally yields topics that feel disjointed or dominated by overly generic terms. Overall, CTM's topics appear more interpretable and semantically consistent than those generated by LDA.

📝❓ Do the two models generate similar topics? Can you discover the same temporal trends (if there are any)? Discuss in 5-6 sentences.

✅ Yes, LDA and CTM mainly generate similar topics. For instance, topics centered on programming languages and set theory dominate before 1990, control theory becomes prominent from 1990-2009, and after 2010 deep learning and optimization emerge as the leading themes.

One notable difference is that CTM identifies health- and biomedicine-related themes that do not appear as distinctly in LDA. In the 1990-2009 period, CTM includes topics like machine learning for biomedicine, and after 2010 it separates out brain imaging.

During topic annotation, we found that CTM yields more refined, narrowly defined topics, whereas LDA tends to merge several subthemes into broader categories. In terms of temporal dynamics, both models reflect major shifts such as the growing prominence of neural networks, control systems, and web technologies, but CTM captures these changes with greater specificity. For example, it separates early neural network themes from later deep learning oriented ones. LDA still indicates overall trends, but with less clarity. Overall, the models align on major patterns but differ in granularity and coherence.

📝❓ Can you suggest an alternate model apart from paraphrase-mpnet-base-v2? What could be some of the possible advantages and disadvantages of using an alternate model? Hint: Look at some of the models [here](https://huggingface.co/spaces/mteb/leaderboard). Note: You do not need to execute the code for an alternate model.

✅ An effective alternative is `all-MiniLM-L6-v2`. On the Hugging Face Embedding Leaderboard, it is among the strongest performers within its size class, making it a practical choice when you want a good balance between quality and efficiency.

A key advantage of switching to it is that it is **smaller and faster**, which can noticeably reduce embedding time and memory usage when working with large corpora. Despite its lighter footprint, its embeddings are typically **strong enough for similarity search and clustering**, so in a CTM pipeline it can still produce reasonably coherent topic structure without the computational overhead of larger transformer encoders.

The main downside is that, because it is a more compact model, it may **miss finer-grained semantic nuances** that larger encoders capture, which can slightly hurt topic quality for highly technical or domain-specific text.

## Lab Report

### 1. Introduction
In this notebook, we explored topic modeling using Latent Dirchlet Allocation (LDA) and the Contextualized Topic Model (CTM). Analysis is conducted scross three time frames: before 1990, 1990 to 2009 and 2010 onwards. For each period, two different preprocessing methods and varying number of topics are appplied to examine their impact on topic quality and coherence. The exercise aims to evaluate how well each model captures meaningful topics and assess the differences in performance between LDA and CTM.


### 2. Methodology

#### 2.1 Preprocessing

Two preprocessing functions were implemented to test how text normalization affects topic modeling. The first preprocessing function performs standard cleaning: converting text to lowercase, removing non-alphabetic characters and removing stopwords. The second preprocessing function builds on this by also applying lemmatization.

#### 2.2 LDA

LDA was applied with 5 and 10 number of topics to observe how topic granularity changes. For each time period, LDA was trained using both preprocessing functions and evaluated qualitatively by inspecting the top keywords from each topic.

#### 2.3 CTM

CTM was also run with 5 and 10 topics using the same preprocessing strategies as LDA, enabling a direct comparison between the two models. Because CTM integrates contextual information, it often produces more coherent and interpretable topics, particularly when documents contain complex or ambiguous terminology.

### 3. Results

Across all periods and preprocessing schemes, CTM consistently produced topics with higher coherence than LDA. LDA topics were generally meaningful, but in some configurations they contained mixed or ambiguous word groups, especially under minimal preprocessing (`preprocess1()`) or when the topic number was large (`num_topics=10` frequently yielded incoherent topics). CTM topics were clearer and more cohesive, often grouping related technical or methodological terms more effectively.

### 4. Discussion

The comparison between LDA and CTM is consistent with Bianchi et al. (2021): CTM often produces more coherent topics because it incorporates contextual embeddings. While LDA can still capture broad themes, it tends to degrade when different topics share many of the same common words. In contrast, CTM can use semantic context to distinguish meanings and preserve topic quality even when word usage looks similar.

Temporal trends, such as the emergence of deep learning, control systems, and data-driven methods post-1990 were identifiable in both models, but CTM presented these trends with clearer boundaries.